# Seedlist — FastHTML + Semantic HTML Rebuild

Recreating [seedlist.com](https://seedlist.com) as a dynamic FastHTML app using semantic custom HTML tags. The original is a static site (Jinja2 + build.py) with ~6400 lines across 21 files. This version achieves feature parity in ~1700 lines (~26% of the original).

**What we have:**
- Full site shell with responsive nav, all pages, all routes
- Investor listing with client-side filtering (stage, sector, location, sort, group)
- Individual profiles for investors, firms, and startups with rendered markdown, footnotes, depth badges
- Investor "Similar Investors" and "Paths to" sections (co-investors, firm colleagues, founders, groups)
- Groups page with curated collections
- Rounds feed
- Three founder tools: Find (target list builder), Comparables (startup picker), Enrich (CSV upload)
- Agents/API documentation page
- JSON API endpoints for client-side tools
- Global search on homepage
- Automatically deployed on solveit

## Roadmap: Founder → Investor Matching Pipeline

**Goal:** Turn Seedlist from a self-serve directory into an active matchmaking service with Eric as the quality gate.

### Auth & Accounts
- Use solveit auth (Gmail sign-in) for user identity
- Collect email as a field in the intake form — no separate verification for now
- Founder's account doesn't store submission history (yet)

### Founder Intake
- Logged-in founders see a banner/CTA inviting them to submit their startup for matching
- Intake form captures structured data: stage, sector, raise amount, location, traction metrics, what they want beyond capital
- CSV upload option remains available for investor lists they already have
- Email field included in the form for delivery

### Matching Pipeline
- On submission, Seedlist's matching agent processes the founder's data against all investor profiles
- Scoring based on: thesis alignment, portfolio conflicts, check size fit, domain value-add
- Produces a ranked list of top investor matches with reasoning

### Dual Email Delivery
- **Founder** receives an analysis email summarizing their matches and fit reasoning
- **Eric** receives the same analysis plus recommended investors

### Eric's Draft Queue (Gmail API)
- For each matched investor, Seedlist creates a **Gmail draft** in Eric's account via the Gmail API
- Each draft is pre-composed as a forwarding email: Eric's personalized note at the top, founder research below
- All drafts for one founder are threaded under a single summary email using `threadId`
- A custom Gmail label (`Seedlist/Pending`) is applied for easy filtering
- Drafts are ordered strongest match first
- Eric reviews each draft — edits if needed, hits Send or deletes
- Personalized notes are tiered: obvious fits get minimal context, less obvious fits get a brief sentence connecting the dots

### Email Tone
- **Obvious fit:** *"This landed in my inbox and immediately made me think of you."*
- **Less obvious fit:** Brief connection to the investor's recent activity or thesis — one sentence max
- Forwarded block contains the AI-generated research, not a founder-written pitch

## Imports

In [ ]:
import yaml, re, mistletoe, json
from dialoghelper.solve_auth import *
from uuid import uuid4
import dialoghelper.solve_auth as _dsa
_dsa._app_url = lambda port: f'https://{json.loads(os.environ["PUBLIC_DOMAINS"])[str(port)]}'
from collections import Counter, defaultdict
from pathlib import Path
from fasthtml.common import *
from fasthtml.jupyter import *
import frontmatter
from fasthtml.components import (App_Shell, Site_Nav, Nav_Links, Nav_Link, Nav_Brand, Menu_Toggle,
    Main_Content, Site_Footer, Page_Hero, Search_Box, Search_Results, Card_Grid, Link_Card,
    Page_Section, Filter_Bar, Filter_Count, Listing_Grid,
    Investor_Card, Card_Header, Card_Name, Card_Firm, Card_Location, Tag_List,
    Profile_Header, Profile_Subtitle, Profile_Date, Depth_Badge,
    Llm_Notice, Profile_Tldr, Meta_Grid, Meta_Item, Meta_Label, Meta_Value,
    Tag_Section, Profile_Body,
    Team_Section, Team_Grid, Team_Card, Team_Role,
    Rounds_Feed, Round_Card, Round_Date, Round_Main, Round_Detail, Round_Investors, Round_Amount,
    Group_Card, Group_Count, Group_Members,
    Tool_Page, Tool_Form, Tool_Results, Result_Table,
    Startup_Search, Startup_Dropdown, Selected_Chips, Search_Chip,
    Paths_Section, Paths_Co_Companies, Paths_Firm_Label, Paths_Firm_Grid,
    Paths_Founder_Company, Paths_Founder_Names, Paths_Groups,
    Similar_Section, Sub_Link, Privacy_Note, Alt_Tools, Name_Plain,
    Upload_Area, Enrich_Stats, Enrich_Actions, Stat_Item, Stat_Number, Stat_Label, Match_Banner)

## Data Loading & Helpers

In [ ]:
DATA = Path('/app/data/seedlist/data')

def process_footnotes(html):
    "Convert [^N] refs to anchor links and [^N]: defs to clickable footnotes."
    html = re.sub(r'\[(\^(\d+))\](?!:)', r'<sup><a href="#fn-\2" id="fnref-\2">\2</a></sup>', html)
    def _fn_def(m):
        num, text = m.group(1), m.group(2).strip()
        url_m = re.search(r'(https?://\S+)', text)
        if url_m:
            url, desc = url_m.group(1), text[:url_m.start()].strip()
            return f'<div class="footnote" id="fn-{num}"><sup>[{num}]</sup> {desc} <a href="{url}" target="_blank" rel="noopener">{url}</a> <a href="#fnref-{num}">↩</a></div>'
        return f'<div class="footnote" id="fn-{num}"><sup>[{num}]</sup> {text} <a href="#fnref-{num}">↩</a></div>'
    html = re.sub(r'\[\^(\d+)\]:\s*(.*?)(?=\[\^\d+\]:|</p>|$)', _fn_def, html, flags=re.DOTALL)
    html = re.sub(r'<p>\s*(<div class="footnote")', r'\1', html)
    return re.sub(r'(</div>)\s*</p>', r'\1', html)

def _depth(raw, ptype):
    "Classify profile quality: Reviewed, Summary, or Stub."
    words, sources = len(raw.split()), len(re.findall(r'^\[\^\d+\]:', raw, re.MULTILINE))
    has_port = '## Portfolio' in raw or '## Funding History' in raw
    has_quotes = bool(re.search(r'## What (Investors|Founders) Say\n\n(?!No )', raw))
    if ptype == 'investor':
        if words >= 500 and sources >= 5 and has_port and '## Inferred Thesis' in raw: return 'Reviewed'
        if words >= 200 and sources >= 2: return 'Summary'
    elif ptype == 'firm':
        if words >= 400 and sources >= 4 and '## About' in raw: return 'Reviewed'
        if words >= 150 and sources >= 2: return 'Summary'
    elif ptype == 'startup':
        if words >= 400 and sources >= 3 and has_port and has_quotes: return 'Reviewed'
        if words >= 200 and sources >= 2 and has_port: return 'Summary'
    return 'Stub'

def _render(post, ptype):
    "Add depth and rendered content to a profile dict."
    p = dict(post.metadata)
    p['profile_depth'] = _depth(post.content, ptype)
    p['content'] = process_footnotes(mistletoe.markdown(post.content))
    return p

def load_profiles(subdir, slug=None):
    "Load published profiles from subdir. If slug given, load one with rendered content."
    ptype = subdir.rstrip('s')
    if slug:
        f = DATA/subdir/f'{slug}.md'
        if not f.exists(): return None
        return _render(frontmatter.load(f), ptype)
    return [dict(post.metadata) for f in sorted((DATA/subdir).glob('*.md'))
            if (post := frontmatter.load(f)) and post.metadata.get('status') == 'published']

def fmt_date(d):
    "Format YYYY-MM-DD to human-readable."
    if not d: return ''
    ms = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    s = str(d).split('-')
    if len(s) == 3: return f"{ms[int(s[1])-1]} {int(s[2])}, {s[0]}"
    if len(s) == 2: return f"{ms[int(s[1])-1]} {s[0]}"
    return s[0]

def _sort_date(s):
    "Parse approximate date strings into sortable YYYY-MM."
    if not s: return ''
    s = s.strip().lstrip('~')
    ms = {'jan':'01','feb':'02','mar':'03','apr':'04','may':'05','jun':'06','jul':'07','aug':'08','sep':'09','oct':'10','nov':'11','dec':'12'}
    m = re.match(r'([A-Za-z]+)\s+(\d{4})', s)
    if m: return f"{m.group(2)}-{ms.get(m.group(1).lower()[:3],'01')}"
    m = re.match(r'(\d{4})-(\d{2})', s)
    if m: return s[:7]
    m = re.match(r'(\d{4})', s)
    if m: return f"{m.group(1)}-01"
    return ''

In [ ]:
def parse_md_table(content, headers):
    "Parse rows from first markdown table whose header contains any of `headers`. Returns list of column lists."
    rows, in_table = [], False
    for line in content.split('\n'):
        if not in_table and any(h in line for h in headers): in_table = True; continue
        if in_table and line.startswith('|---'): continue
        if in_table and line.startswith('|'):
            cols = [c.strip() for c in line.split('|')[1:-1]]
            if cols: rows.append(cols)
        elif in_table and not line.strip(): in_table = False
    return rows

def _load_rounds():
    "Parse funding tables from startup profiles into sorted rounds list."
    rounds = []
    for f in sorted((DATA/'startups').glob('*.md')):
        post = frontmatter.load(f)
        meta = dict(post.metadata)
        if meta.get('status') != 'published': continue
        slug, name, sectors = meta.get('slug',''), meta.get('name',''), meta.get('sector', [])
        for cols in parse_md_table(post.content, ['| Round']):
            if len(cols) >= 3:
                rounds.append(dict(company=name, company_slug=slug, round=cols[0], date=cols[1],
                    sort_date=_sort_date(cols[1]), amount=cols[2], lead=cols[3] if len(cols)>3 else '', sectors=sectors))
    rounds.sort(key=lambda r: r['sort_date'], reverse=True)
    return rounds

def _load_taxonomy():
    "Load sector taxonomy and build tag→parent mapping."
    with open(DATA/'sector-taxonomy.yaml') as f: tax = yaml.safe_load(f)
    t2p = {}
    for parent, info in tax.items():
        for tag in info.get('tags', []): t2p[tag.lower()] = parent
    return tax, t2p

def _load_clusters():
    "Load cluster data and build investor→groups mapping."
    with open(DATA/'clusters.json') as f: cd = json.load(f)
    grps = {}
    for c in cd.get('curated_collections', []):
        for m in c.get('members', []): grps.setdefault(m['slug'], []).append(c['name'])
    return cd, grps

investors = load_profiles('investors')
firms = load_profiles('firms')
startups = load_profiles('startups')
inv_lookup = {p['slug']: p for p in investors}
firm_lookup = {p['slug']: p for p in firms}
startup_lookup = {str(p.get('slug','')): p for p in startups}
all_rounds = _load_rounds()
taxonomy, tag_to_parent = _load_taxonomy()
parent_labels = {k: v['label'] for k,v in taxonomy.items()}
clusters_data, inv_groups = _load_clusters()
curated = clusters_data.get('curated_collections', [])

len(investors), len(firms), len(startups), len(all_rounds)

(564, 179, 267, 993)

## CSS

In [ ]:
css = """
/* reset & variables */
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
:root {
  --bg: #ffffff; --text: #1a1a1a; --muted: #6b7280;
  --border: #e5e7eb; --accent: #2563eb; --accent-light: #eff6ff;
  --surface: #f9fafb; --radius: 6px; --max-w: 960px;
}

/* base */
html { font-size: 16px; -webkit-font-smoothing: antialiased; }
body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; color: var(--text); background: var(--bg); line-height: 1.6; min-height: 100vh; display: flex; flex-direction: column; }
a { color: var(--accent); text-decoration: none; }
a:hover { text-decoration: underline; }

/* cards */
investor-card, link-card, group-card, team-card { display: block; border: 1px solid var(--border); border-radius: var(--radius); color: var(--text); transition: border-color 0.15s, box-shadow 0.15s; }
investor-card:hover, link-card:hover, group-card:hover { border-color: var(--accent); box-shadow: 0 2px 8px rgba(37,99,235,0.08); text-decoration: none; }

/* pills */
depth-badge, tag-list a, tag-list span, meta-label, stat-label, .sector-btn, search-chip { border-radius: 999px; font-weight: 600; text-transform: uppercase; letter-spacing: 0.04em; }

/* focus */
select:focus, input:focus { border-color: var(--accent); box-shadow: 0 0 0 3px rgba(37,99,235,0.1); outline: none; }

/* form inputs */
tool-form select, tool-form input[type="text"], tool-form textarea, startup-search input { width: 100%; padding: 0.6rem 0.75rem; font-size: 0.9rem; border: 1px solid var(--border); border-radius: var(--radius); outline: none; }

/* headings */
profile-body h2, paths-section h2, similar-section h2, team-section h2, tool-results .results-header h2 { font-size: 1.25rem; font-weight: 700; margin-bottom: 0.75rem; }
profile-body h2, paths-section h2, similar-section h2 { border-bottom: 1px solid var(--border); padding-bottom: 0.35rem; }

/* muted small */
site-footer, card-firm, filter-count, sub-link, paths-firm-label, paths-founder-names, alt-tools,
group-card group-members, paths-firm-grid a, #group-banner, round-detail, llm-notice, .clear-btn { font-size: 0.85rem; color: var(--muted); }

/* shell */
app-shell { display: flex; flex-direction: column; min-height: 100vh; }

/* nav */
site-nav {
  display: flex; align-items: center; justify-content: space-between; height: 56px;
  max-width: var(--max-w); width: 100%; margin: 0 auto; padding: 0 1.25rem;
  border-bottom: 1px solid var(--border); position: sticky; top: 0; background: var(--bg); z-index: 100;
}
nav-brand {
  font-size: 1.25rem; font-weight: 700; color: var(--text); letter-spacing: -0.02em;
  & a { color: inherit; }
  & a:hover { text-decoration: none; }
}
nav-links { display: flex; gap: 1.5rem; list-style: none; }
nav-link {
  display: inline;
  & a { color: var(--muted); font-size: 0.9rem; font-weight: 500; }
  & a:hover { color: var(--text); text-decoration: none; }
}
menu-toggle { display: none; background: none; border: none; cursor: pointer; padding: 0.5rem; font-size: 1.5rem; color: var(--text); line-height: 1; }

/* layout */
main-content { flex: 1; max-width: var(--max-w); width: 100%; margin: 0 auto; padding: 2.5rem 1.25rem; }
site-footer { border-top: 1px solid var(--border); padding: 1.5rem 0; margin-top: 2rem; display: block; text-align: center; }
page-section {
  display: block;
  & h1 { font-size: 1.75rem; font-weight: 700; letter-spacing: -0.02em; margin-bottom: 1rem; }
}

/* hero & search */
page-hero {
  display: block; text-align: center; padding: 3rem 0 2rem;
  & h1 { font-size: 2rem; font-weight: 700; letter-spacing: -0.03em; margin-bottom: 0.75rem; }
  & p { color: var(--muted); font-size: 1.1rem; margin-bottom: 2rem; }
}
search-box {
  display: block; position: relative; max-width: 520px; margin: 0 auto;
  & input { width: 100%; padding: 0.75rem 1rem; font-size: 1rem; border: 1px solid var(--border); border-radius: var(--radius); outline: none; background: var(--bg); transition: border-color 0.15s; }
}
search-results {
  display: block; position: absolute; top: 100%; left: 0; right: 0; background: var(--bg);
  border: 1px solid var(--border); border-top: none; border-radius: 0 0 var(--radius) var(--radius);
  max-height: 400px; overflow-y: auto; z-index: 50; box-shadow: 0 4px 12px rgba(0,0,0,0.06);
  &[hidden] { display: none; }
  & a:hover { background: var(--surface); text-decoration: none; }
}

/* card grid */
card-grid {
  display: grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)); gap: 1.25rem; margin-top: 1rem;
  & > a { text-decoration: none; color: inherit; }
}
link-card {
  padding: 1.5rem; text-align: center;
  & h2 { font-size: 1.25rem; margin-bottom: 0.25rem; }
  & p { color: var(--muted); font-size: 0.9rem; }
}

/* filters & listing */
filter-bar {
  display: flex; gap: 0.75rem; flex-wrap: wrap; margin-bottom: 0.75rem;
  & select { padding: 0.55rem 0.75rem; font-size: 0.9rem; border: 1px solid var(--border); border-radius: var(--radius); background: var(--bg); color: var(--text); outline: none; cursor: pointer; }
}
.clear-btn { padding: 0.55rem 1rem; border: 1px solid var(--border); border-radius: var(--radius); background: var(--surface); cursor: pointer; transition: color 0.15s, border-color 0.15s; }
.clear-btn:hover { color: var(--text); border-color: var(--text); }
filter-count { display: block; margin-bottom: 1rem; }
listing-grid {
  display: grid; grid-template-columns: repeat(auto-fill, minmax(280px, 1fr)); gap: 1rem;
  & > a { display: flex; color: inherit; text-decoration: none !important; }
}

/* investor cards */
investor-card { display: flex; flex-direction: column; flex: 1; padding: 1rem; }
card-header { display: block; margin-bottom: 0.25rem; }
card-name { display: block; font-weight: 600; font-size: 0.95rem; }
card-firm { display: block; }
card-location { display: block; font-size: 0.82rem; color: var(--muted); margin-bottom: 0.4rem; }
tag-list {
  display: flex; flex-wrap: wrap; gap: 0.35rem;
  & a, & span { display: inline-block; font-size: 0.65rem; padding: 0.15rem 0.5rem; }
  & [stage] { background: #f5f3ff; color: #7c3aed; }
  & [sector] { background: #ecfeff; color: #0891b2; }
}

/* profile */
profile-header {
  display: block; margin-bottom: 1rem;
  & h1 { font-size: 1.75rem; font-weight: 700; letter-spacing: -0.02em; margin-bottom: 0.25rem; }
}
profile-subtitle { display: block; font-size: 1rem; color: var(--muted); margin-bottom: 0.5rem; }
depth-badge {
  display: inline-block; font-size: 0.72rem; padding: 0.15rem 0.5rem; cursor: help;
  &[level="reviewed"] { background: #d1fae5; color: #065f46; }
  &[level="summary"] { background: #fef3c7; color: #92400e; }
  &[level="stub"] { background: #fee2e2; color: #991b1b; }
}
profile-date { font-size: 0.8rem; color: var(--muted); margin-left: 0.5rem; }
llm-notice { display: block; background: var(--accent-light); border-radius: var(--radius); padding: 0.75rem 1rem; margin-bottom: 1.5rem; }
profile-tldr { display: block; font-size: 1rem; line-height: 1.7; margin-bottom: 1.5rem; padding: 1rem; background: var(--surface); border-radius: var(--radius); }
meta-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(200px, 1fr)); gap: 0.75rem; margin-bottom: 1.5rem; }
meta-item { display: block; }
meta-label { display: block; font-size: 0.75rem; color: var(--muted); margin-bottom: 0.15rem; }
meta-value { display: block; font-size: 0.95rem; }
tag-section { display: block; margin-bottom: 1rem; }
profile-body {
  display: block; line-height: 1.7;
  & h2 { margin: 1.5rem 0 0.75rem; }
  & h3 { font-size: 1.05rem; font-weight: 600; margin: 1.25rem 0 0.5rem; }
  & p { margin-bottom: 0.75rem; }
  & table { width: 100%; border-collapse: collapse; margin: 1rem 0; font-size: 0.85rem; }
  & th, & td { text-align: left; padding: 0.5rem 0.65rem; border-bottom: 1px solid var(--border); }
  & th { font-weight: 600; font-size: 0.75rem; text-transform: uppercase; letter-spacing: 0.03em; color: var(--muted); }
  & sup { font-size: 0.7em; }
  & .footnote { font-size: 0.8rem; color: var(--muted); }
  & pre { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 1rem; overflow-x: auto; font-size: 0.85rem; line-height: 1.5; margin-bottom: 0.75rem; }
  & code { font-family: "SF Mono", Menlo, Monaco, "Courier New", monospace; font-size: 0.85em; }
  & p code { background: var(--surface); padding: 0.15rem 0.35rem; border-radius: 3px; }
  & ul, & ol { padding-left: 1.5rem; margin-bottom: 0.75rem; }
  & li { margin-bottom: 0.35rem; }
}

/* team */
team-section { display: block; margin: 1.5rem 0; }
team-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(200px, 1fr)); gap: 0.75rem; }
team-card {
  padding: 0.75rem;
  & a { font-weight: 600; font-size: 0.95rem; }
}
team-role { display: block; font-size: 0.82rem; color: var(--muted); margin-top: 0.15rem; }

/* rounds */
rounds-feed { display: block; max-width: 720px; margin: 0 auto; }
round-card { display: flex; align-items: baseline; gap: 1rem; padding: 0.75rem 0; border-bottom: 1px solid var(--border); }
round-date { flex-shrink: 0; width: 100px; font-size: 0.8rem; color: var(--muted); text-align: right; }
round-main {
  flex: 1; display: block;
  & a { font-weight: 600; font-size: 0.95rem; }
}
round-detail { display: block; margin-top: 0.15rem; }
round-investors { display: block; font-size: 0.8rem; color: var(--muted); margin-top: 0.15rem; }
round-amount { flex-shrink: 0; font-weight: 600; font-size: 0.9rem; color: var(--accent); }

/* groups */
group-card {
  padding: 1.25rem 1.5rem; margin-bottom: 1rem;
  & h2 { font-size: 1.15rem; font-weight: 700; margin-bottom: 0.35rem; }
  & h2 a { color: var(--text); }
  & h2 a:hover { color: var(--accent); text-decoration: none; }
  & p { font-size: 0.9rem; color: var(--muted); margin-bottom: 0.5rem; }
  & group-members { line-height: 1.6; }
}
group-count { display: block; font-size: 0.82rem; font-weight: 600; color: var(--muted); margin-bottom: 0.35rem; }

/* tools */
tool-page { display: block; max-width: 720px; margin: 0 auto; }
tool-form {
  display: block; margin-bottom: 2rem;
  & label { display: block; font-size: 0.85rem; color: var(--text); font-weight: 600; margin-bottom: 0.25rem; }
  & select, & input[type="text"] { background: var(--bg); color: var(--text); margin-bottom: 1rem; }
  & .sector-grid { display: flex; flex-wrap: wrap; gap: 0.4rem; margin-bottom: 1rem; }
  & .sector-btn { display: inline-block; font-size: 0.75rem; padding: 0.3rem 0.65rem; border: 1px solid var(--border); background: var(--bg); color: var(--muted); cursor: pointer; transition: all 0.15s; }
  & .sector-btn:hover { border-color: var(--accent); color: var(--accent); }
  & .sector-btn[active] { background: var(--accent); color: white; border-color: var(--accent); }
  & .hint { font-size: 0.8rem; color: var(--muted); margin-top: -0.75rem; margin-bottom: 1rem; }
  & textarea { resize: vertical; font-family: inherit; line-height: 1.5; }
  & input[type="file"] { width: 100%; padding: 0.6rem 0.75rem; font-size: 0.9rem; border: 1px solid var(--border); border-radius: var(--radius); background: var(--bg); color: var(--text); cursor: pointer; margin-bottom: 1rem; }
  & input[type="file"]::file-selector-button { padding: 0.35rem 0.85rem; font-size: 0.82rem; font-weight: 600; border: 1px solid var(--border); border-radius: var(--radius); background: var(--surface); color: var(--text); cursor: pointer; margin-right: 0.75rem; transition: all 0.15s; }
  & input[type="file"]::file-selector-button:hover { border-color: var(--accent); color: var(--accent); }
  & button[type="submit"] { padding: 0.7rem 2rem; font-size: 0.95rem; font-weight: 600; background: var(--accent); color: white; border: none; border-radius: var(--radius); cursor: pointer; transition: background 0.15s; }
  & button[type="submit"]:hover { background: #1d4ed8; }
}
tool-results {
  display: none;
  &[visible] { display: block; }
  & .results-header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 1rem; }
  & .results-actions { display: flex; gap: 0.5rem; }
  & .results-actions button { padding: 0.45rem 0.85rem; font-size: 0.82rem; font-weight: 600; border: 1px solid var(--border); border-radius: var(--radius); background: var(--bg); color: var(--text); cursor: pointer; transition: all 0.15s; }
  & .results-actions button:hover { border-color: var(--accent); color: var(--accent); }
}
result-table {
  display: block; width: 100%; overflow-x: auto;
  & table { width: 100%; border-collapse: collapse; font-size: 0.85rem; }
  & th { text-align: left; padding: 0.5rem 0.65rem; border-bottom: 2px solid var(--border); font-weight: 600; font-size: 0.75rem; text-transform: uppercase; letter-spacing: 0.03em; color: var(--muted); }
  & td { padding: 0.5rem 0.65rem; border-bottom: 1px solid var(--border); }
  & tr:hover { background: var(--surface); }
  & .tier-1 { border-left: 3px solid #22c55e; }
  & .tier-2 { border-left: 3px solid #f59e0b; }
  & .tier-3 { border-left: 3px solid var(--border); }
  & a { font-weight: 600; }
}

/* startup picker */
startup-search { position: relative; margin-bottom: 1rem; }
startup-dropdown {
  display: none; position: absolute; top: 100%; left: 0; right: 0; max-height: 200px; overflow-y: auto;
  background: var(--bg); border: 1px solid var(--border); border-top: none;
  border-radius: 0 0 var(--radius) var(--radius); z-index: 50; box-shadow: 0 4px 12px rgba(0,0,0,0.06);
  &[open] { display: block; }
  & div { padding: 0.5rem 0.75rem; cursor: pointer; font-size: 0.9rem; }
  & div:hover { background: var(--surface); }
}
selected-chips {
  display: flex; flex-wrap: wrap; gap: 0.4rem; margin-bottom: 1rem;
  & search-chip { display: inline-flex; align-items: center; gap: 0.3rem; font-size: 0.8rem; padding: 0.25rem 0.6rem; background: var(--accent-light); color: var(--accent); }
  & search-chip button { background: none; border: none; cursor: pointer; font-size: 0.9rem; color: var(--muted); padding: 0; line-height: 1; }
}

/* paths & similar */
#group-banner { display: none; align-items: center; gap: 0.5rem; padding: 0.5rem 0.75rem; background: var(--accent-light); border-radius: var(--radius); margin-bottom: 0.75rem; }
privacy-note { font-size: 0.8rem; color: var(--muted); margin-top: 0.75rem; }
name-plain { font-weight: 600; font-size: 0.95rem; }
paths-section {
  margin-top: 2rem;
  & h2 { margin-bottom: 0.5rem; }
  & > p { color: var(--muted); font-size: 0.9rem; margin-bottom: 1.5rem; }
  & h3 { font-size: 0.8rem; font-weight: 600; color: var(--muted); text-transform: uppercase; letter-spacing: 0.03em; margin: 1.5rem 0 0.5rem; }
  & ul { list-style: none; padding: 0; }
  & li { padding: 0.5rem 0; border-bottom: 1px solid var(--border); }
  & li:last-child { border-bottom: none; }
  & li a { font-weight: 600; font-size: 0.95rem; }
}
paths-co-companies { font-size: 0.82rem; color: var(--muted); margin-top: 0.1rem; }
paths-firm-label { margin-bottom: 0.25rem; }
paths-firm-grid {
  display: flex; flex-wrap: wrap; gap: 0.4rem;
  & a { display: inline-block; font-weight: 500; padding: 0.2rem 0.6rem; border: 1px solid var(--border); border-radius: 999px; transition: border-color 0.15s, color 0.15s; }
  & a:hover { border-color: var(--accent); color: var(--accent); text-decoration: none; }
}
paths-founder-company { font-weight: 600; font-size: 0.95rem; }
paths-groups {
  font-size: 0.9rem;
  & a { font-weight: 600; }
}
similar-section {
  margin-top: 2rem;
  & > p { font-size: 0.9rem; margin-bottom: 0.75rem; }
  & > p a { font-weight: 600; }
}

/* upload & enrich */
upload-area {
  display: block; border: 2px dashed var(--border); border-radius: var(--radius); padding: 3rem 2rem; text-align: center; cursor: pointer; transition: border-color 0.15s, background 0.15s; margin: 1.5rem 0;
  &:hover, &.drag-over { border-color: var(--accent); background: var(--accent-light); }
  & p { color: var(--muted); font-size: 0.95rem; margin: 0.5rem 0; }
  & .icon { font-size: 2rem; display: block; margin-bottom: 0.5rem; }
  & .cta { color: var(--accent); font-weight: 600; }
  & input[type="file"] { display: none; }
}
enrich-stats { display: flex; gap: 1.5rem; justify-content: center; flex-wrap: wrap; margin: 1rem 0 1.5rem; }
stat-item {
  text-align: center; display: block;
  &[kind="exact"] stat-number { color: #059669; }
  &[kind="fuzzy"] stat-number { color: #d97706; }
  &[kind="none"] stat-number { color: #dc2626; }
}
stat-number { display: block; font-size: 1.5rem; font-weight: 700; }
stat-label { display: block; font-size: 0.75rem; color: var(--muted); }
.match-exact { color: #059669; font-weight: 600; }
.match-fuzzy { color: #d97706; font-weight: 600; }
.match-none { color: #dc2626; }
enrich-actions {
  display: flex; gap: 1rem; justify-content: center; margin: 1.5rem 0;
  & button { padding: 0.6rem 1.5rem; border-radius: var(--radius); font-size: 0.9rem; font-weight: 600; cursor: pointer; border: none; transition: background 0.15s; }
  & .btn-primary { background: var(--accent); color: white; }
  & .btn-primary:hover { background: #1d4ed8; }
  & .btn-secondary { background: var(--surface); color: var(--text); border: 1px solid var(--border); }
  & .btn-secondary:hover { background: var(--border); }
}

/* responsive */
match-banner {
  display: flex; align-items: center; justify-content: center; gap: 1rem;
  padding: 0.75rem 1.25rem; background: linear-gradient(135deg, #eff6ff, #f5f3ff);
  border-bottom: 1px solid var(--border); font-size: 0.9rem; flex-wrap: wrap;
}
match-banner p { margin: 0; }
.banner-btn {
  display: inline-block; padding: 0.4rem 1rem; background: var(--accent); color: white;
  border-radius: var(--radius); font-weight: 600; font-size: 0.85rem; white-space: nowrap;
}
.banner-btn:hover { background: #1d4ed8; text-decoration: none; }

/* responsive */
@media (max-width: 768px) {
  match-banner { flex-direction: column; text-align: center; gap: 0.5rem; }
  page-hero h1 { font-size: 1.5rem; }
  page-hero p { font-size: 0.95rem; }
  menu-toggle { display: block; }
  nav-links { display: none; position: absolute; top: 56px; left: 0; right: 0; flex-direction: column; background: var(--bg); border-bottom: 1px solid var(--border); padding: 0.75rem 1.25rem; gap: 0.5rem; box-shadow: 0 4px 12px rgba(0,0,0,0.06); z-index: 99; }
  nav-links[open] { display: flex; }
  nav-link a { font-size: 1rem; padding: 0.35rem 0; display: block; }
  round-card { flex-wrap: wrap; }
  round-date { width: auto; text-align: left; }
}
"""

## App Shell & Setup

In [ ]:
nav_items = [('/investors', 'Investors'), ('/groups', 'Groups'), ('/firms', 'Firms'),
             ('/startups', 'Startups'), ('/rounds', 'Rounds'), ('/discover', 'For Founders'), ('/agents', 'For Agents')]

hamburger_js = Script("document.querySelector('menu-toggle').onclick = e => { let n = document.querySelector('nav-links'); n.toggleAttribute('open'); e.currentTarget.textContent = n.hasAttribute('open') ? '✕' : '☰'; }")

def Shell(*c, title='Seedlist'):
    return (Title(f'{title} — Seedlist.com'),
            App_Shell(
                Site_Nav(Nav_Brand(A('Seedlist', href='/')),
                        Nav_Links(*[Nav_Link(A(label, href=href)) for href,label in nav_items],
                            Nav_Link(Span(hx_get='/auth-status', hx_trigger='load', hx_swap='outerHTML'))),
                        Menu_Toggle('☰', role='button', aria_label='Toggle menu')),
                Div(hx_get='/auth-banner', hx_trigger='load', hx_swap='innerHTML'),
                Main_Content(*c),
                Site_Footer(P('Seedlist.com — LLM-researched investor directory')),
                hamburger_js))

app,rt = fast_app(pico=False, hdrs=[Style(css)], static_path='/app/data/seedlist/static')
setup_solve_signin(app)

@rt('/auth-status')
def get(session):
    if session.get('auth'): return A('Sign out', href='/sign-out')
    return A('Sign in', href=solve_signin_rt)

@rt
def signin_completed(session, signin_reply: str=''):
    session['auth'] = sub_from_signin(session, signin_reply)
    return RedirectResponse('/', status_code=303)

@rt('/sign-out')
def get(session):
    session.pop('auth', '')
    return RedirectResponse('/', status_code=303)

@rt('/auth-banner')
def get(session):
    if not session.get('auth'): return ''
    return Match_Banner(
        P('🚀 ', Strong('Get matched with investors.'), ' Submit your startup and we\'ll find the best fits.'),
        A('Submit your startup →', href='/match', cls='banner-btn'),
        role='banner')

## Component Helpers

In [ ]:
top_stages = ['pre-seed', 'seed', 'series-a', 'series-b', 'growth']
sector_counts = Counter(s for p in investors for s in (p.get('sector_focus') or []))
top_sectors = [s for s,_ in sector_counts.most_common(20)]

_loc_rules = [
    ('sf-bay-area', ['san francisco','menlo park','palo alto','mountain view','woodside','redwood city','atherton','bay area',
                     'saratoga','cupertino','sunnyvale','san mateo','portola valley','los altos','burlingame','hillsborough']),
    ('nyc', ['new york','brooklyn']),
    ('la', ['los angeles','santa monica','venice','beverly hills']),
    ('boston', ['boston','cambridge, ma','somerville, ma']),
    ('seattle', ['seattle','bellevue','kirkland','redmond']),
    ('austin', ['austin']), ('chicago', ['chicago']), ('miami', ['miami','fort lauderdale']),
    ('dc', ['washington, dc','washington, d.c.','washington dc']),
    ('denver', ['denver','boulder']), ('dallas', ['dallas','fort worth']),
    ('san-diego', ['san diego']), ('philadelphia', ['philadelphia']),
    ('portland', ['portland, or','portland, oregon']),
    ('salt-lake-city', ['salt lake','park city',', utah',', ut']),
    ('pittsburgh', ['pittsburgh']), ('atlanta', ['atlanta']), ('minneapolis', ['minneapolis','st. paul']),
    ('london', ['london','england']), ('paris', ['paris']), ('berlin', ['berlin']),
    ('sweden', ['stockholm','malmö','malmo','sweden']),
    ('amsterdam', ['amsterdam','netherlands']), ('switzerland', ['zurich','geneva','switzerland']),
    ('germany', ['munich','germany']), ('estonia', ['tallinn','estonia']), ('ireland', ['ireland','dublin']),
    ('singapore', ['singapore']), ('israel', ['tel aviv','israel','jerusalem']),
    ('india', ['india','bangalore','bengaluru','mumbai','new delhi']),
    ('japan', ['tokyo','japan']), ('hong-kong', ['hong kong']),
    ('china', ['beijing','shanghai','china']), ('uae', ['dubai','uae','abu dhabi']),
    ('canada', ['toronto','vancouver','montreal','ottawa','canada']),
    ('brazil', ['são paulo','sao paulo','brazil','rio']),
    ('australia', ['sydney','melbourne','australia']),
    ('us-other', ['california','texas','florida','georgia','virginia','north carolina','colorado','ohio','michigan',
                  'arizona','tennessee','maryland','connecticut','new jersey',
                  ', ca',', tx',', fl',', ga',', va',', nc',', co',', oh',', mi',', az',', tn',', md',', ct',', nj',', pa',', il',', wa',', or',', mn',', mo']),
]

def loc_region(loc):
    "Map a location string to a region slug."
    if not loc: return ''
    l = loc.lower()
    for region, keywords in _loc_rules:
        if any(kw in l for kw in keywords): return region
    return ''

def investor_sectors(p):
    "Map an investor's sector_focus tags to parent taxonomy keys."
    parents = set()
    for s in (p.get('sector_focus') or []):
        par = tag_to_parent.get(s.lower())
        if par: parents.add(par)
    return ','.join(parents)

loc_groups = {
    'US': [('sf-bay-area','SF Bay Area'),('nyc','New York'),('la','Los Angeles'),('boston','Boston'),('seattle','Seattle'),
           ('austin','Austin'),('chicago','Chicago'),('miami','Miami'),('dc','Washington DC'),('denver','Denver / Boulder'),
           ('dallas','Dallas'),('san-diego','San Diego'),('philadelphia','Philadelphia'),('portland','Portland'),
           ('salt-lake-city','Salt Lake City'),('pittsburgh','Pittsburgh'),('atlanta','Atlanta'),('minneapolis','Minneapolis'),('us-other','US — Other')],
    'Europe': [('london','London'),('paris','Paris'),('berlin','Berlin'),('sweden','Sweden'),('amsterdam','Amsterdam'),
               ('switzerland','Switzerland'),('germany','Germany'),('estonia','Estonia'),('ireland','Ireland')],
    'Asia & Middle East': [('singapore','Singapore'),('israel','Israel'),('india','India'),('japan','Japan'),
                           ('hong-kong','Hong Kong'),('china','China'),('uae','UAE')],
    'Other': [('canada','Canada'),('brazil','Brazil'),('australia','Australia')],
}
active_locs = {loc_region(p.get('location','')) for p in investors} - {''}

def stage_options():    return [Option('All Stages', value='')] + [Option(s.replace('-',' ').title(), value=s) for s in top_stages]
def sector_options():   return [Option('All Sectors', value='')] + [Option(parent_labels[k], value=k) for k in taxonomy]
def sort_options():     return [Option('Recently Active', value='recent'), Option('Name (A-Z)', value='name-asc'), Option('Name (Z-A)', value='name-desc')]

def location_options():
    "Build location dropdown with optgroups, only showing regions that have investors."
    opts = [Option('All Locations', value='')]
    for group, locs in loc_groups.items():
        active = [(slug, label) for slug, label in locs if slug in active_locs]
        if active: opts.append(Optgroup(*[Option(label, value=slug) for slug, label in active], label=group))
    return opts

def depth_badge(level):
    "Render a profile depth badge."
    if not level: return ''
    tips = dict(Reviewed='Multiple independent sources, cited claims, and substantive analysis',
                Summary='Partial profile with limited sources — more research needed',
                Stub='Minimal data only — not yet researched')
    return Depth_Badge(level, level=level.lower(), title=tips.get(level,''))

def profile_meta(*items): return Meta_Grid(*[Meta_Item(Meta_Label(k), Meta_Value(v)) for k,v in items if v])

def social_links(soc):
    "Render social media links for a profile."
    if not soc: return ''
    links = []
    if soc.get('twitter'):  links.append(A(f"@{soc['twitter'].lstrip('@')}", href=f"https://twitter.com/{soc['twitter'].lstrip('@')}", target='_blank'))
    if soc.get('bluesky'):  links.append(A(soc['bluesky'], href=f"https://bsky.app/profile/{soc['bluesky'].lstrip('@')}", target='_blank'))
    if soc.get('linkedin'): links.append(A('LinkedIn', href=soc['linkedin'] if 'http' in soc['linkedin'] else f"https://{soc['linkedin']}", target='_blank'))
    return Span(*[i for pair in zip(links, [' ']*len(links)) for i in pair][:-1])

def investor_cards():
    "Build the list of investor card elements for the listing grid."
    cards = []
    for p in investors:
        tags = [Span(s, stage='') for s in (p.get('stage_focus') or [])]
        region = loc_region(p.get('location',''))
        lvi = p.get('last_verified_investment')
        last_active = str(lvi.get('date','')).lstrip('~') if isinstance(lvi, dict) else ''
        cards.append(A(Investor_Card(
            Card_Header(Card_Name(p.get('name','')), Card_Firm(p.get('firm','')) if p.get('firm') else ''),
            Card_Location(p.get('location','')) if p.get('location') else '',
            Tag_List(*tags) if tags else ''),
            href=f"/investors/{p.get('slug','')}",
            data_stages=','.join(p.get('stage_focus') or []),
            data_sectors=investor_sectors(p),
            data_location=region, data_last_active=last_active, data_name=p.get('name',''),
            data_groups=','.join(inv_groups.get(p.get('slug',''), []))))
    return cards

## Routes — Listings & Profiles

In [ ]:
filter_js = Script("""
(function() {
  var grid = document.getElementById('investor-grid');
  var cards = Array.from(grid.querySelectorAll('a'));
  var countEl = document.getElementById('visible-count');
  var selects = Array.from(document.querySelectorAll('filter-bar select'));
  var groupBanner = document.getElementById('group-banner');
  var groupNameEl = document.getElementById('group-name-display');
  var activeGroup = '';

  function val(id) { return document.getElementById(id).value; }

  var params = new URLSearchParams(window.location.search);
  if (params.get('group')) {
    activeGroup = params.get('group');
    if (groupBanner) { groupBanner.style.display = 'flex'; groupNameEl.textContent = activeGroup; }
  }

  function applyFilters() {
    var stage = val('filter-stage'), sector = val('filter-sector'), loc = val('filter-location'), n = 0;
    cards.forEach(function(c) {
      var show = true;
      if (stage && c.dataset.stages.indexOf(stage) === -1) show = false;
      if (sector && c.dataset.sectors.indexOf(sector) === -1) show = false;
      if (loc && c.dataset.location !== loc) show = false;
      if (activeGroup && c.dataset.groups.indexOf(activeGroup) === -1) show = false;
      c.style.display = show ? '' : 'none';
      if (show) n++;
    });
    countEl.textContent = n;
  }

  function applySort() {
    var mode = val('filter-sort');
    cards.sort(function(a, b) {
      if (mode === 'name-asc') return a.dataset.name.localeCompare(b.dataset.name);
      if (mode === 'name-desc') return b.dataset.name.localeCompare(a.dataset.name);
      var da = a.dataset.lastActive || '', db = b.dataset.lastActive || '';
      if (!da && !db) return a.dataset.name.localeCompare(b.dataset.name);
      if (!da) return 1; if (!db) return -1;
      return db.localeCompare(da);
    });
    cards.forEach(function(c) { grid.appendChild(c); });
  }

  selects.forEach(function(s) { s.addEventListener('change', function() { applySort(); applyFilters(); }); });

  var clearBtn = document.getElementById('clear-filters');
  clearBtn.addEventListener('click', function() {
    selects.forEach(function(s) { s.value = s.options[0].value; });
    activeGroup = '';
    if (groupBanner) groupBanner.style.display = 'none';
    history.replaceState(null, '', '/investors');
    applySort(); applyFilters();
  });

  if (activeGroup) { applySort(); applyFilters(); }
})();
""")

@rt('/investors')
def get():
    group_banner = Div(Span('Filtering by group: '), Strong('', id='group-name-display'),
id='group-banner')
    return Shell(
        Page_Section(
            H1('All Investors'),
            group_banner,
            Filter_Bar(Select(*stage_options(), id='filter-stage'),
                       Select(*sector_options(), id='filter-sector'),
                       Select(*location_options(), id='filter-location'),
                       Select(*sort_options(), id='filter-sort'),
                       Button('Clear', id='clear-filters', cls='clear-btn')),
            Filter_Count('Showing ', Span(str(len(investors)), id='visible-count'), f' of {len(investors)} investors'),
            Listing_Grid(*investor_cards(), id='investor-grid'),
            filter_js),
        title='All Investors')

@rt('/groups')
def get():
    cards = []
    for c in curated:
        members = c.get('members', [])
        member_links = [A(m['name'], href=f"/investors/{m['slug']}") for m in members]
        member_sep = []
        for i,link in enumerate(member_links):
            if i: member_sep.append(', ')
            member_sep.append(link)
        cards.append(Group_Card(
            H2(A(c['name'], href=f"/investors?group={c['name']}")),
            P(c.get('tagline','')) if c.get('tagline') else '',
            Group_Count(f'{len(members)} investors'),
            Group_Members(*member_sep)))
    return Shell(
        Page_Hero(H1('Investor Groups'), P(f'{len(investors)} investors organized by era, thesis, and investment pattern. Investors can appear in multiple groups.')),
        *cards,
        title='Groups')


@rt('/rounds')
def get():
    cards = []
    for r in all_rounds[:100]:
        detail = ' · '.join([x for x in [r['round'], ', '.join(r['sectors'][:2])] if x])
        cards.append(Round_Card(
            Round_Date(r['date']),
            Round_Main(A(r['company'], href=f"/startups/{r['company_slug']}"),
                       Round_Detail(detail) if detail else '',
                       Round_Investors(r['lead']) if r['lead'] else ''),
            Round_Amount(r['amount']) if r['amount'] and r['amount'] != '—' else ''))
    return Shell(
        Page_Hero(H1('Round Feed'), P('Recent startup funding rounds from our database.')),
        Rounds_Feed(*cards),
        title='Round Feed')

@rt('/firms')
def get():
    cards = []
    for p in firms:
        tags = [Span(s, stage='') for s in (p.get('stage_focus') or [])]
        cards.append(A(Investor_Card(
            Card_Header(Card_Name(p.get('name','')), Card_Firm(p.get('location','')) if p.get('location') else ''),
            Tag_List(*tags) if tags else ''),
            href=f"/firms/{p.get('slug','')}"))
    return Shell(
        Page_Section(H1('All Firms'), Filter_Count(f'{len(firms)} firms'),
            Listing_Grid(*cards)),
        title='All Firms')

@rt('/firms/{slug}')
def get(slug: str):
    p = load_profiles('firms', slug)
    if not p: return Shell(Page_Section(H1('Not Found'), P('Firm not found.')), title='Not Found')
    team_cards = []
    for m in (p.get('team') or []):
        s = m.get('slug','')
        name = m.get('name') or inv_lookup.get(s,{}).get('name') or s.replace('-',' ').title()
        link = A(name, href=f"/investors/{s}") if s in inv_lookup else Name_Plain(name)
        team_cards.append(Team_Card(link, Team_Role(m.get('role','')) if m.get('role') else ''))
    return Shell(
        Article(
            Profile_Header(
                H1(p['name']),
                depth_badge(p.get('profile_depth','')),
                Profile_Date(f"Updated {fmt_date(p.get('last_researched',''))}")),
            Llm_Notice(P('This profile is AI-generated. Help us improve it by sharing corrections.')),
            profile_meta(('Location', p.get('location')), ('Founded', p.get('founded')),
                         ('Fund Size', p.get('fund_size')), ('Website', A(p['website'], href=p['website'], target='_blank') if p.get('website') else None)),
            Tag_Section(Meta_Label('Stage Focus'), Tag_List(*[A(s, href=f'/investors?stage={s}', stage='') for s in (p.get('stage_focus') or [])])) if p.get('stage_focus') else '',
            Tag_Section(Meta_Label('Sector Focus'), Tag_List(*[A(s, href=f'/investors?sector={s}', sector='') for s in (p.get('sector_focus') or [])])) if p.get('sector_focus') else '',
            Team_Section(H2('Team'), Team_Grid(*team_cards)) if team_cards else '',
            Profile_Body(Safe(p.get('content','')))),
        title=p['name'])

@rt('/startups')
def get():
    cards = []
    for p in startups:
        tags = [Span(s, sector='') for s in (p.get('sector') or [])]
        cards.append(A(Investor_Card(
            Card_Header(Card_Name(p.get('name','')), Card_Firm(p.get('stage_latest','')) if p.get('stage_latest') else ''),
            Card_Location(p.get('location','')) if p.get('location') else '',
            Tag_List(*tags) if tags else ''),
            href=f"/startups/{p.get('slug','')}"))
    return Shell(
        Page_Section(H1('All Startups'), Filter_Count(f'{len(startups)} startups'),
            Listing_Grid(*cards)),
        title='All Startups')

@rt('/startups/{slug}')
def get(slug: str):
    p = load_profiles('startups', slug)
    if not p: return Shell(Page_Section(H1('Not Found'), P('Startup not found.')), title='Not Found')
    inv_cards = []
    for inv in (p.get('investors') or []):
        s = inv.get('slug') or ''
        if not s: continue
        name = inv_lookup.get(s,{}).get('name') or s.replace('-',' ').title()
        link = A(name, href=f"/investors/{s}") if s in inv_lookup else Name_Plain(name)
        role = inv.get('round','') + (f" ({inv.get('year','')})" if inv.get('year') else '')
        inv_cards.append(Team_Card(link, Team_Role(role) if role else ''))
    for f in (p.get('firms') or []):
        s = f.get('slug') or ''
        if not s: continue
        name = firm_lookup.get(s,{}).get('name') or s.replace('-',' ').title()
        link = A(name, href=f"/firms/{s}") if s in firm_lookup else Name_Plain(name)
        role = f.get('round','') + (f" ({f.get('year','')})" if f.get('year') else '')
        inv_cards.append(Team_Card(link, Team_Role(role) if role else ''))
    founder_cards = [Team_Card(Name_Plain(fo.get('name','')), Team_Role(fo.get('role','')) if fo.get('role') else '') for fo in (p.get('founders') or [])]
    return Shell(
        Article(
            Profile_Header(
                H1(p['name']),
                depth_badge(p.get('profile_depth','')),
                Profile_Date(f"Updated {fmt_date(p.get('last_researched',''))}")),
            Llm_Notice(P('This profile is AI-generated. Help us improve it by sharing corrections.')),
            profile_meta(('Location', p.get('location')), ('Founded', p.get('founded')),
                         ('Latest Stage', p.get('stage_latest')), ('Total Raised', p.get('total_raised')),
                         ('Website', A(p['website'], href=p['website'], target='_blank') if p.get('website') else None)),
            Tag_Section(Meta_Label('Sector'), Tag_List(*[A(s, href=f'/investors?sector={s}', sector='') for s in (p.get('sector') or [])])) if p.get('sector') else '',
            Team_Section(H2('Investors'), Team_Grid(*inv_cards)) if inv_cards else '',
            Team_Section(H2('Founders'), Team_Grid(*founder_cards)) if founder_cards else '',
            Profile_Body(Safe(p.get('content','')))),
        title=p['name'])

## Routes — Investor Detail (Similar & Paths)

In [ ]:
similar_map = clusters_data.get('similar_investors', {})
inv_cluster_map = clusters_data.get('investor_clusters', {})
algo_clusters = clusters_data.get('algo_clusters', clusters_data.get('clusters', []))
cluster_by_id = {c['id']: c for c in algo_clusters}


inv_companies = defaultdict(set)
for f in sorted((DATA/'investors').glob('*.md')):
    post = frontmatter.load(f)
    meta = dict(post.metadata)
    if meta.get('status') != 'published': continue
    slug = meta.get('slug', '')
    if not slug: continue
    for cols in parse_md_table(post.content, ['| Company', '| Portfolio']):
        company = re.sub(r'\[([^\]]+)\].*', r'\1', cols[0]).strip().lower().replace(' ','-').replace('.','').replace("'",'')
        if company: inv_companies[slug].add(company)

for s in startups:
    s_slug = str(s.get('slug',''))
    for i in (s.get('investors') or []):
        if i.get('slug'): inv_companies[i['slug']].add(s_slug)

company_to_invs = defaultdict(set)
for inv_slug, companies in inv_companies.items():
    for co in companies: company_to_invs[co].add(inv_slug)

co_invest = defaultdict(lambda: defaultdict(list))
for company, inv_set in company_to_invs.items():
    inv_list = sorted(inv_set)
    for i,a in enumerate(inv_list):
        for b in inv_list[i+1:]:
            co_invest[a][b].append(company)
            co_invest[b][a].append(company)

def similar_section(slug):
    "Build the Similar Investors section for a given investor slug."
    sims = similar_map.get(slug, [])[:5]
    if not sims: return ''
    groups = inv_groups.get(slug, [])
    group_links = [A(g, href=f"/investors?group={g}") for g in groups]
    group_parts = []
    for i,l in enumerate(group_links):
        if i: group_parts.append(' ')
        group_parts.append(l)
    sim_cards = []
    for s in sims:
        si = s if isinstance(s, str) else s.get('slug', '')
        p = inv_lookup.get(si)
        if not p: continue
        tags = [Span(st, stage='') for st in (p.get('stage_focus') or [])]
        sim_cards.append(A(Investor_Card(
            Card_Header(Card_Name(p.get('name','')), Card_Firm(p.get('firm','')) if p.get('firm') else ''),
            Card_Location(p.get('location','')) if p.get('location') else '',
            Tag_List(*tags) if tags else ''),
            href=f"/investors/{si}"))
    return Similar_Section(H2('Similar Investors'), P(*group_parts) if group_parts else '', Listing_Grid(*sim_cards))

def paths_section(slug):
    "Build the full Paths to [investor] section with co-investors, firm colleagues, founders, and groups."
    name = inv_lookup.get(slug, {}).get('name', slug)
    parts = []

    peers = co_invest.get(slug, {})
    if peers:
        top = sorted(peers.items(), key=lambda x: -len(x[1]))[:10]
        items = []
        for peer_slug, companies in top:
            if len(companies) < 2: continue
            pi = inv_lookup.get(peer_slug)
            pname = pi.get('name','') if pi else peer_slug.replace('-',' ').title()
            firm = f" ({pi.get('firm','')})" if pi and pi.get('firm') and pi['firm'] != 'independent' else ''
            link = A(f"{pname}{firm}", href=f"/investors/{peer_slug}") if pi else Name_Plain(f"{pname}{firm}")
            items.append(Li(link, f' — {len(companies)} shared investments', Paths_Co_Companies(', '.join(companies[:4]))))
        if items: parts.append(Div(H3('Through co-investors'), Ul(*items)))

    firm_slug = inv_lookup.get(slug, {}).get('firm', '')
    if firm_slug:
        colleagues = [p for p in investors if p.get('firm') == firm_slug and p.get('slug') != slug]
        if colleagues:
            chips = [A(c.get('name',''), href=f"/investors/{c.get('slug','')}") for c in colleagues[:15]]
            parts.append(Div(H3('Through firm colleagues'), Paths_Firm_Label(f'At {firm_slug}:'), Paths_Firm_Grid(*chips)))

    founder_items = []
    for s in startups:
        s_slug = str(s.get('slug',''))
        backer_slugs = [i.get('slug','') for i in (s.get('investors') or [])] + [f.get('slug','') for f in (s.get('firms') or [])]
        if slug in backer_slugs and s.get('founders'):
            fnames = ', '.join(f"{f.get('name','')} ({f.get('role','')})" if f.get('role') else f.get('name','') for f in s['founders'])
            founder_items.append(Li(Paths_Founder_Company(A(s.get('name',''), href=f"/startups/{s_slug}")), Paths_Founder_Names(fnames)))
    if founder_items:
        parts.append(Div(H3('Through portfolio company founders'), P('Founders of companies this investor backed — they can make a warm intro.', style='font-size:0.85rem;color:var(--muted);margin-bottom:0.5rem'), Ul(*founder_items)))

    groups = inv_groups.get(slug, [])
    if groups:
        glinks = []
        for i,g in enumerate(groups):
            if i: glinks.append(', ')
            glinks.append(A(g, href=f"/investors?group={g}"))
        parts.append(Div(H3('Through groups'), Paths_Groups('Member of: ', *glinks)))

    if not parts: return ''
    return Paths_Section(H2(f'Paths to {name}'), P('Ways to get a warm introduction, based on shared investments and professional connections.'), *parts)

@rt('/investors/{slug}')
def get(slug: str):
    p = load_profiles('investors', slug)
    if not p: return Shell(Page_Section(H1('Not Found'), P('Investor profile not found.')), title='Not Found')
    lvi = p.get('last_verified_investment')
    lvi_text = ''
    if isinstance(lvi, dict):
        lvi_text = f"{lvi.get('company','')} ({lvi.get('round','')}) — {fmt_date(lvi.get('date',''))}"
    return Shell(
        Article(
            Profile_Header(
                H1(p['name']),
                Profile_Subtitle(f"{p.get('role','')} at ", A(p.get('firm',''), href=f"/firms/{p.get('firm','')}")) if p.get('role') and p.get('firm') else '',
                depth_badge(p.get('profile_depth','')),
                Profile_Date(f"Updated {fmt_date(p.get('last_researched',''))}")),
            Llm_Notice(P('This profile is AI-generated. Help us improve it by sharing corrections.')),
            Profile_Tldr(P(p.get('tldr',''))) if p.get('tldr') else '',
            profile_meta(('Location', p.get('location')), ('Check Size', p.get('check_size')),
                         ('Last Investment', lvi_text), ('Social', social_links(p.get('social')))),
            Tag_Section(Meta_Label('Stage Focus'), Tag_List(*[A(s, href=f'/investors?stage={s}', stage='') for s in (p.get('stage_focus') or [])])) if p.get('stage_focus') else '',
            Tag_Section(Meta_Label('Sector Focus'), Tag_List(*[A(s, href=f'/investors?sector={s}', sector='') for s in (p.get('sector_focus') or [])])) if p.get('sector_focus') else '',
            Profile_Body(Safe(p.get('content',''))),
            similar_section(slug),
            paths_section(slug)),
        title=p['name'])

## Routes — Agents & API

In [ ]:
agents_md = """
# Seedlist for AI Agents

Structured investor data for AI agents helping founders raise money. No auth, no rate limits — just JSON.

{stats}

## Quick Start

All endpoints return static JSON. No API key needed. Fetch with curl, Python, or any HTTP client. Data is rebuilt daily; check `last_updated` field on the root object for freshness.

**Tip:** For quick lookups by slug, use the lightweight `/investor-lookup.json` (~150KB). For full search/filter across all investors, use `/enrichment-index.json` (~400KB).

### curl
```
curl -s https://seedlist.com/enrichment-index.json | python3 -m json.tool | head -50
```

### Python
```python
import json, urllib.request

data = json.loads(urllib.request.urlopen("https://seedlist.com/enrichment-index.json").read())

matches = [
    inv for inv in data["investors"]
    if "seed" in inv["stage_focus"]
    and "fintech" in [s.lower() for s in inv["sector_focus"]]
]

for inv in matches[:5]:
    summary = inv.get("tldr") or inv["thesis_summary"][:80]
    firm = inv["firm_name"] or inv["firm"]
    print(f"{inv['name']} ({firm}) — {summary}")
```

See Data Notes at the bottom for field-level details and edge cases. Or read [llms.txt](/llms.txt) for a machine-readable summary.

## Endpoints

| Endpoint | URL | Description |
|----------|-----|-------------|
| Investor Index | `/enrichment-index.json` | All investors and firms with stage, sector, check size, location, thesis summary (~400KB) |
| Investor Lookup | `/investor-lookup.json` | Lightweight slug-keyed dictionary for O(1) investor lookups (~150KB) |
| Investor Graph | `/investor-graph.json` | Co-investment relationships, firm memberships, startup founders |
| Startup-Investor Map | `/startup-investor-map.json` | Which investors backed which startups, with round and year |
| Rounds Feed | `/rounds-feed.json` | 500 most recent funding rounds with company, date, amount, lead |
| Search Index | `/search-index.json` | Searchable index of all investors, firms, and startups |
| Cluster Data | `/cluster-data.json` | 40 curated investor collections plus algorithmic clusters |

## Common Tasks

### 1. Build a fundraising target list

Score investors by stage match (30pts), sector overlap weighted by specificity (40pts), recency (15pts), check size fit (15pts). Group into 3 tiers. Target: 10-20 in Tier 1, 15-30 in Tier 2.

```python
import json, urllib.request

data = json.loads(urllib.request.urlopen("https://seedlist.com/enrichment-index.json").read())

my_stage = "seed"
my_sectors = ["fintech", "developer-tools"]

results = []
for inv in data["investors"]:
    score, reasons = 0, []
    if my_stage in inv["stage_focus"]:
        score += 30
        reasons.append(f"Invests at {my_stage}")
    inv_sectors = [s.lower() for s in inv["sector_focus"]]
    overlap = [s for s in my_sectors if s in inv_sectors]
    if overlap:
        specificity = min(1.0, 5 / max(len(inv_sectors), 1))
        sector_pts = int(min(40, len(overlap) * 15) * (0.5 + 0.5 * specificity))
        score += sector_pts
        reasons.append(f"Sectors: {', '.join(overlap)}")
    last = inv.get("last_active", "")
    if last >= "2025": score += 15
    elif last >= "2024": score += 8
    if score >= 30:
        tier = 1 if score >= 70 else 2 if score >= 45 else 3
        results.append({"name": inv["name"], "firm": inv["firm_name"] or inv["firm"],
            "tier": tier, "why": "; ".join(reasons)})

results.sort(key=lambda x: x["tier"])
for r in results:
    label = ["Strong fit", "Worth pursuing", "Stretch"][r["tier"] - 1]
    print(f"[{label}] {r['name']} ({r['firm']}) — {r['why']}")
```

### 2. Find investors by comparable companies

Look up which investors backed similar startups. Deduplicate by slug per startup.

```python
import json, urllib.request
from collections import defaultdict

data = json.loads(urllib.request.urlopen("https://seedlist.com/startup-investor-map.json").read())
comparables = ["stripe", "plaid", "brex"]

investor_companies = defaultdict(lambda: {"name": "", "companies": set()})
for slug in comparables:
    seen = set()
    for inv in data["startup_investors"].get(slug, []):
        if inv["slug"] in seen or inv["slug"] == "independent": continue
        seen.add(inv["slug"])
        entry = investor_companies[inv["slug"]]
        entry["name"] = inv["name"]
        entry["companies"].add(slug)

results = sorted(investor_companies.values(), key=lambda x: -len(x["companies"]))
for r in results[:15]:
    print(f"{r['name']}: backed {len(r['companies'])}/{len(comparables)} ({', '.join(r['companies'])})")
```

### 3. Find warm intro paths to an investor

```python
import requests
graph = requests.get("https://seedlist.com/investor-graph.json").json()
target = "ron-conway"

co = graph["co_investments"].get(target, {})
for slug, info in sorted(co.items(), key=lambda x: -x[1]["count"])[:10]:
    name = graph["investor_names"].get(slug, slug)
    print(f"Co-investor: {name} ({info['count']} shared deals)")

for startup_slug, backers in graph["startup_backers"].items():
    if target in backers:
        founders = graph["startup_founders"].get(startup_slug, [])
        name = graph["startup_names"].get(startup_slug, startup_slug)
        for f in founders:
            print(f"Founder intro: {f['name']} ({f['role']} at {name})")
```

### 4. Enrich an investor CSV

```python
import requests
data = requests.get("https://seedlist.com/enrichment-index.json").json()

by_name = {}
for inv in data["investors"]:
    by_name[inv["name"].lower()] = inv

my_investors = ["Ron Conway", "Mike Maples", "Unknown Person"]
for name in my_investors:
    match = by_name.get(name.lower())
    if match:
        print(f"MATCH: {match['name']} | {match['firm_name']} | Stage: {match['stage_focus']}")
    else:
        print(f"NO MATCH: {name}")
```

### 5. Track recent funding rounds

```python
import requests
rounds = requests.get("https://seedlist.com/rounds-feed.json").json()
for r in rounds[:10]:
    amt = r["amount"] or "undisclosed"
    print(f"{r['date']} | {r['company']} | {r['round']} | {amt} | Lead: {r['lead']}")
```

## Claude Skill

If you use [Claude Code](https://claude.com/claude-code), install a Seedlist skill that teaches Claude how to query these endpoints.

```
curl -o seedlist-skill.md https://raw.githubusercontent.com/ericries/seedlist/main/docs/seedlist-skill.md
cp seedlist-skill.md .claude/skills/seedlist-skill.md
```

## Data Notes

- **Sectors are normalized lowercase strings** — direct string matching is safe. Use `tldr` for summaries (95% populated).
- **`firm` is a slug, `firm_name` is display.** `firm_name` may be empty — fall back: `inv["firm_name"] or inv["firm"]`
- **`last_active` format varies** — "2025-11-15", "2025-03", "~2022", or empty. String comparison works for recency.
- **`check_size` is human-readable** — e.g., "$250K-$2M". Parse with regex for range comparison.
- **Startup-investor map has duplicates** — deduplicate by slug when counting comparables. Filter out `"independent"`.
- **`queued` array** — investors in the research pipeline, not yet profiled.

---

[llms.txt (machine-readable summary)](/llms.txt) · [Claude Skill file](https://github.com/ericries/seedlist/blob/main/docs/seedlist-skill.md) · [GitHub source](https://github.com/ericries/seedlist)
"""

@rt('/agents')
def get():
    stats = f'{len(investors)} investors · {len(firms)} firms · {len(startups)} startups'
    html = process_footnotes(mistletoe.markdown(agents_md.replace('{stats}', stats)))
    return Shell(Profile_Body(Safe(html)), title='For AI Agents')

## Routes — Founder Tools

In [ ]:
import json as _json

@rt('/api/investors.json')
def get():
    data = []
    for p in investors:
        lvi = p.get('last_verified_investment')
        last_active = str(lvi.get('date','')).lstrip('~') if isinstance(lvi, dict) else ''
        data.append(dict(name=p.get('name',''), slug=p.get('slug',''), firm=p.get('firm',''),
            location=p.get('location',''), check_size=p.get('check_size',''), tldr=p.get('tldr',''),
            stage_focus=p.get('stage_focus',[]), sector_focus=p.get('sector_focus',[]),
            last_active=last_active, sectors_parent=investor_sectors(p),
            region=loc_region(p.get('location',''))))
    return Response(_json.dumps(data), media_type='application/json')

@rt('/api/startup-investors.json')
def get():
    data = {}
    for s in startups:
        backers = []
        for i in (s.get('investors') or []):
            sl = i.get('slug') or ''
            if not sl: continue
            name = inv_lookup.get(sl,{}).get('name','') or sl.replace('-',' ').title()
            backers.append(dict(slug=sl, name=name, round=i.get('round',''), year=i.get('year',''), type='investor'))
        for f in (s.get('firms') or []):
            sl = f.get('slug') or ''
            if not sl: continue
            name = firm_lookup.get(sl,{}).get('name','') or sl.replace('-',' ').title()
            backers.append(dict(slug=sl, name=name, round=f.get('round',''), year=f.get('year',''), type='firm'))
        if backers: data[str(s.get('slug',''))] = backers
    return Response(_json.dumps(data), media_type='application/json')

@rt('/api/startups.json')
def get():
    data = [dict(name=s.get('name',''), slug=str(s.get('slug','')), sector=s.get('sector',[])) for s in startups]
    return Response(_json.dumps(data), media_type='application/json')

find_js = Script("""
(function() {
  var data = null, selectedSectors = new Set();
  var form = document.getElementById('find-form');
  var results = document.getElementById('find-results');
  var tbody = document.getElementById('find-tbody');
  var countEl = document.getElementById('result-count');
  var sectorBtns = document.querySelectorAll('.sector-btn');
  var hintEl = document.getElementById('sector-hint');

  sectorBtns.forEach(function(btn) {
    btn.addEventListener('click', function() {
      var v = btn.dataset.value;
      if (selectedSectors.has(v)) { selectedSectors.delete(v); btn.removeAttribute('active'); }
      else if (selectedSectors.size < 3) { selectedSectors.add(v); btn.setAttribute('active',''); }
      hintEl.textContent = selectedSectors.size + '/3 selected.';
    });
  });

  document.getElementById('find-submit').addEventListener('click', async function(e) {
    if (!data) { var r = await fetch('/api/investors.json'); data = await r.json(); }
    var stage = document.getElementById('f-stage').value;
    var check = document.getElementById('f-check').value;
    var loc = document.getElementById('f-location').value;
    var sectors = Array.from(selectedSectors);

    var scored = [];
    data.forEach(function(inv) {
      var score = 0, reasons = [];
      if (stage && inv.stage_focus.indexOf(stage) !== -1) { score += 30; reasons.push('Stage: ' + stage); }
      if (sectors.length) {
        var invSec = inv.sector_focus.map(function(s){return s.toLowerCase();});
        var overlap = sectors.filter(function(s){return invSec.indexOf(s) !== -1;});
        if (overlap.length) {
          var spec = Math.min(1.0, 5 / Math.max(invSec.length, 1));
          score += Math.round(Math.min(40, overlap.length * 15) * (0.5 + 0.5 * spec));
          reasons.push('Sectors: ' + overlap.join(', '));
        }
      }
      if (inv.last_active >= '2025') score += 15;
      else if (inv.last_active >= '2024') score += 8;
      if (check && inv.check_size) {
        var nums = inv.check_size.match(/[\\d.]+/g);
        if (nums) {
          var lo = parseFloat(nums[0]), hi = nums.length > 1 ? parseFloat(nums[nums.length-1]) : lo;
          if (inv.check_size.indexOf('M') !== -1) { if (lo < 100) lo *= 1000; if (hi < 100) hi *= 1000; }
          var ranges = {'lt500':[0,500],'500-2m':[250,2000],'2m-5m':[1000,5000],'5m+':[3000,999999]};
          var r = ranges[check];
          if (r && hi >= r[0] && lo <= r[1]) { score += 15; reasons.push('Check size fits'); }
        }
      }
      if (loc && loc !== '') {
        if (inv.region === loc) { score += 5; reasons.push('Location match'); }
      }
      if (score >= 25) {
        var tier = score >= 65 ? 1 : score >= 40 ? 2 : 3;
        scored.push({inv:inv, score:score, tier:tier, reasons:reasons.join('; ')});
      }
    });

    scored.sort(function(a,b){ return a.tier !== b.tier ? a.tier - b.tier : b.score - a.score; });
    tbody.innerHTML = '';
    scored.forEach(function(r) {
      var labels = ['Strong fit','Worth pursuing','Stretch'];
      var tr = document.createElement('tr');
      tr.className = 'tier-' + r.tier;
      tr.innerHTML = '<td><a href="/investors/' + r.inv.slug + '">' + r.inv.name + '</a></td>'
        + '<td>' + (r.inv.firm || '—') + '</td>'
        + '<td>' + labels[r.tier-1] + '</td>'
        + '<td>' + r.inv.stage_focus.join(', ') + '</td>'
        + '<td>' + (r.inv.check_size || '—') + '</td>'
        + '<td style="font-size:0.8rem;color:var(--muted)">' + r.reasons + '</td>';
      tbody.appendChild(tr);
    });
    countEl.textContent = scored.length;
    results.setAttribute('visible','');
  });

  document.getElementById('find-reset').addEventListener('click', function() {
    results.removeAttribute('visible');
    form.reset();
    selectedSectors.clear();
    sectorBtns.forEach(function(b){b.removeAttribute('active');});
    hintEl.textContent = '0/3 selected.';
  });

  document.getElementById('find-csv').addEventListener('click', function() {
    var rows = [['Name','Firm','Tier','Stages','Check Size','Why']];
    tbody.querySelectorAll('tr').forEach(function(tr) {
      var cells = tr.querySelectorAll('td');
      rows.push(Array.from(cells).map(function(c){return '"' + c.textContent.replace(/"/g,'""') + '"';}));
    });
    var blob = new Blob([rows.map(function(r){return r.join(',');}).join('\\n')], {type:'text/csv'});
    var a = document.createElement('a'); a.href = URL.createObjectURL(blob);
    a.download = 'seedlist-target-list.csv'; a.click();
  });
})();
""")

sector_btns = [Span(parent_labels[k], cls='sector-btn', data_value=k) for k in taxonomy]

@rt('/find')
def get():
    return Shell(
        Tool_Page(
            Page_Hero(H1('Build your investor target list'),
                     P(f'Answer a few questions and we\'ll find the best-fit investors from our database of {len(investors)} profiles.'),
                     Sub_Link(A('Part of Founder Tools', href='/discover'))),
            Tool_Form(
                Label('Stage *', _for='f-stage'),
                Select(Option('Select stage...', value=''), Option('Pre-seed', value='pre-seed'),
                       Option('Seed', value='seed'), Option('Series A', value='series-a'),
                       Option('Series B+', value='series-b'), id='f-stage'),
                Label('Sectors *'),
                Div(*sector_btns, cls='sector-grid'),
                P('Pick 1 to 3 sectors. ', Span('0/3 selected.', id='sector-hint'), cls='hint'),
                Label('Check size needed', _for='f-check'),
                Select(Option('No preference', value=''), Option('< $500K', value='lt500'),
                       Option('$500K – $2M', value='500-2m'), Option('$2M – $5M', value='2m-5m'),
                       Option('$5M+', value='5m+'), id='f-check'),
                Label('Location preference', _for='f-location'),
                Select(Option('No preference', value=''), Option('SF Bay Area', value='sf-bay-area'),
                       Option('NYC', value='nyc'), Option('Other US', value='us-other'),
                       Option('International', value=''), id='f-location'),
                Button('Find investors', id='find-submit'),
                Privacy_Note('Your data never leaves your browser.'),
                id='find-form'),
            Tool_Results(
                Div(H2(Span('0', id='result-count'), ' investors found'),
                    Div(Button('Download as CSV', id='find-csv'), Button('Start over', id='find-reset'), cls='results-actions'),
                    cls='results-header'),
                Result_Table(Table(Thead(Tr(Th('Name'), Th('Firm'), Th('Tier'), Th('Stages'), Th('Check'), Th('Why'))),
                                   Tbody(id='find-tbody'))),
                Alt_Tools('Try other tools: ', A('Find by comparable companies', href='/comparables'), ' | ', A('Enrich your list', href='/enrich')),
                id='find-results'),
            find_js),
        title='Build Target List')

comp_js = Script("""
(function() {
  var startupData = null, investorMap = null;
  var selected = new Map();
  var input = document.getElementById('comp-input');
  var dropdown = document.getElementById('comp-dropdown');
  var chips = document.getElementById('comp-chips');
  var results = document.getElementById('comp-results');
  var tbody = document.getElementById('comp-tbody');
  var countEl = document.getElementById('comp-count');

  async function loadData() {
    if (!startupData) {
      var r1 = await fetch('/api/startups.json'), r2 = await fetch('/api/startup-investors.json');
      startupData = await r1.json(); investorMap = await r2.json();
    }
  }

  input.addEventListener('input', async function() {
    await loadData();
    var q = input.value.toLowerCase().trim();
    dropdown.innerHTML = '';
    if (q.length < 2) { dropdown.removeAttribute('open'); return; }
    var matches = startupData.filter(function(s) { return s.name.toLowerCase().indexOf(q) !== -1 && !selected.has(s.slug); }).slice(0, 8);
    matches.forEach(function(s) {
      var d = document.createElement('div');
      d.textContent = s.name;
      d.addEventListener('click', function() { addStartup(s); input.value = ''; dropdown.removeAttribute('open'); });
      dropdown.appendChild(d);
    });
    if (matches.length) dropdown.setAttribute('open',''); else dropdown.removeAttribute('open');
  });

  input.addEventListener('blur', function() { setTimeout(function(){ dropdown.removeAttribute('open'); }, 200); });

  function addStartup(s) {
    if (selected.size >= 5 || selected.has(s.slug)) return;
    selected.set(s.slug, s.name);
    renderChips();
  }

  function renderChips() {
    chips.innerHTML = '';
    selected.forEach(function(name, slug) {
      var span = document.createElement('search-chip');
      span.innerHTML = name + ' <button data-slug="' + slug + '">✕</button>';
      span.querySelector('button').addEventListener('click', function() { selected.delete(slug); renderChips(); });
      chips.appendChild(span);
    });
  }

  document.getElementById('comp-submit').addEventListener('click', async function() {
    await loadData();
    if (selected.size === 0) return;
    var slugs = Array.from(selected.keys());
    var investorCounts = {};

    slugs.forEach(function(slug) {
      var backers = investorMap[slug] || [];
      var seen = new Set();
      backers.forEach(function(b) {
        if (seen.has(b.slug)) return;
        seen.add(b.slug);
        if (!investorCounts[b.slug]) investorCounts[b.slug] = {name: b.name, type: b.type, companies: new Set(), rounds: []};
        investorCounts[b.slug].companies.add(slug);
        investorCounts[b.slug].rounds.push(b.round + ' in ' + selected.get(slug));
      });
    });

    var scored = Object.entries(investorCounts).map(function(e) {
      return {slug: e[0], name: e[1].name, type: e[1].type, count: e[1].companies.size, total: slugs.length, rounds: e[1].rounds};
    });
    scored.sort(function(a, b) { return b.count - a.count || a.name.localeCompare(b.name); });

    tbody.innerHTML = '';
    scored.forEach(function(r) {
      var tr = document.createElement('tr');
      var link = r.type === 'investor' ? '/investors/' + r.slug : '/firms/' + r.slug;
      tr.innerHTML = '<td><a href="' + link + '">' + r.name + '</a></td>'
        + '<td>' + (r.type === 'firm' ? 'Firm' : 'Individual') + '</td>'
        + '<td>' + r.count + '/' + r.total + '</td>'
        + '<td style="font-size:0.8rem;color:var(--muted)">' + r.rounds.join(', ') + '</td>';
      tbody.appendChild(tr);
    });
    countEl.textContent = scored.length;
    results.setAttribute('visible','');
  });

  document.getElementById('comp-reset').addEventListener('click', function() {
    results.removeAttribute('visible');
    selected.clear(); renderChips();
    input.value = '';
  });

  document.getElementById('comp-csv').addEventListener('click', function() {
    var rows = [['Name','Type','Match','Rounds']];
    tbody.querySelectorAll('tr').forEach(function(tr) {
      var cells = tr.querySelectorAll('td');
      rows.push(Array.from(cells).map(function(c){ return '"' + c.textContent.replace(/"/g,'""') + '"'; }));
    });
    var blob = new Blob([rows.map(function(r){return r.join(',');}).join('\\n')], {type:'text/csv'});
    var a = document.createElement('a'); a.href = URL.createObjectURL(blob);
    a.download = 'seedlist-comparables.csv'; a.click();
  });
})();
""")

@rt('/comparables')
def get():
    return Shell(
        Tool_Page(
            Page_Hero(H1('Find investors by comparable companies'),
                     P('Pick startups similar to yours — we\'ll show you who invested in them.'),
                     Sub_Link(A('Part of Founder Tools', href='/discover'))),
            Tool_Form(
                Label('Search for startups (up to 5)'),
                Startup_Search(Input(type='text', id='comp-input', placeholder='Type a startup name...', autocomplete='off'),
                    Startup_Dropdown(id='comp-dropdown')),
                Selected_Chips(id='comp-chips'),
                Button('Find investors', id='comp-submit'),
                Privacy_Note('Your data never leaves your browser.')),
            Tool_Results(
                Div(H2(Span('0', id='comp-count'), ' investors found'),
                    Div(Button('Download as CSV', id='comp-csv'), Button('Start over', id='comp-reset'), cls='results-actions'),
                    cls='results-header'),
                Result_Table(Table(Thead(Tr(Th('Name'), Th('Type'), Th('Match'), Th('Rounds'))),
                                   Tbody(id='comp-tbody'))),
                Alt_Tools('Try other tools: ', A('Build a target list', href='/find'), ' | ', A('Enrich your list', href='/enrich')),
                id='comp-results'),
            comp_js),
        title='Find by Comparables')

In [ ]:
enrich_js = Script("""
(function() {
  var apiData = null, byName = {}, bySlug = {};
  var uploadEl = document.getElementById('upload-area');
  var fileInput = document.getElementById('file-input');
  var uploadSection = document.getElementById('upload-section');
  var previewSection = document.getElementById('enrich-preview');
  var statsEl = document.getElementById('enrich-stats');
  var tbody = document.getElementById('enrich-tbody');
  var enrichedRows = [];

  async function loadData() {
    if (apiData) return;
    var r = await fetch('/api/investors.json');
    apiData = await r.json();
    apiData.forEach(function(inv) {
      byName[inv.name.toLowerCase()] = inv;
      bySlug[inv.slug] = inv;
    });
  }

  function normalize(s) { return s.toLowerCase().replace(/[^a-z0-9]/g, ''); }

  function fuzzyMatch(name) {
    var norm = normalize(name);
    if (!norm) return null;
    for (var i = 0; i < apiData.length; i++) {
      if (normalize(apiData[i].name) === norm) return apiData[i];
    }
    for (var i = 0; i < apiData.length; i++) {
      var n = normalize(apiData[i].name);
      if (n.indexOf(norm) !== -1 || norm.indexOf(n) !== -1) return apiData[i];
    }
    return null;
  }

  function parseCSV(text) {
    var lines = text.trim().split(/\\r?\\n/);
    if (lines.length < 2) return [];
    var sep = lines[0].indexOf('\\t') !== -1 ? '\\t' : ',';
    var headers = lines[0].split(sep).map(function(h) { return h.replace(/^"|"$/g, '').trim(); });
    var nameCol = -1;
    for (var i = 0; i < headers.length; i++) {
      var h = headers[i].toLowerCase();
      if (h === 'name' || h === 'investor' || h === 'investor name' || h === 'contact' || h === 'person') { nameCol = i; break; }
    }
    if (nameCol === -1) nameCol = 0;
    var rows = [];
    for (var i = 1; i < lines.length; i++) {
      var cols = lines[i].split(sep).map(function(c) { return c.replace(/^"|"$/g, '').trim(); });
      if (cols[nameCol]) rows.push(cols[nameCol]);
    }
    return rows;
  }

  async function processFile(file) {
    await loadData();
    var text = await file.text();
    var names = parseCSV(text);
    if (!names.length) { alert('No names found in CSV.'); return; }

    var exact = 0, fuzzy = 0, none = 0;
    enrichedRows = [];
    tbody.innerHTML = '';

    names.forEach(function(name) {
      var inv = byName[name.toLowerCase()];
      var matchType = 'exact';
      if (!inv) { inv = fuzzyMatch(name); matchType = inv ? 'fuzzy' : 'none'; }
      if (inv) {
        if (matchType === 'exact') exact++; else fuzzy++;
        var tr = document.createElement('tr');
        tr.innerHTML = '<td>' + name + '</td>'
          + '<td class="match-' + matchType + '">' + matchType + '</td>'
          + '<td><a href="/investors/' + inv.slug + '">' + inv.name + '</a></td>'
          + '<td>' + (inv.firm || '—') + '</td>'
          + '<td>' + inv.stage_focus.join(', ') + '</td>'
          + '<td>' + (inv.check_size || '—') + '</td>'
          + '<td style="max-width:200px;font-size:0.8rem;color:var(--muted);white-space:normal">' + (inv.tldr || '').slice(0, 120) + '</td>';
        tbody.appendChild(tr);
        enrichedRows.push([name, matchType, inv.name, inv.firm, inv.stage_focus.join('; '), inv.sector_focus.join('; '), inv.check_size, inv.location, inv.tldr]);
      } else {
        none++;
        var tr = document.createElement('tr');
        tr.innerHTML = '<td>' + name + '</td><td class="match-none">none</td><td colspan="5">—</td>';
        tbody.appendChild(tr);
        enrichedRows.push([name, 'none', '', '', '', '', '', '', '']);
      }
    });

    statsEl.innerHTML = '<stat-item kind="exact"><stat-number>' + exact + '</stat-number><stat-label>Exact</stat-label></stat-item>'
      + '<stat-item kind="fuzzy"><stat-number>' + fuzzy + '</stat-number><stat-label>Fuzzy</stat-label></stat-item>'
      + '<stat-item kind="none"><stat-number>' + none + '</stat-number><stat-label>Not found</stat-label></stat-item>';

    uploadSection.style.display = 'none';
    previewSection.style.display = 'block';
  }

  uploadEl.addEventListener('click', function() { fileInput.click(); });
  fileInput.addEventListener('change', function() { if (fileInput.files[0]) processFile(fileInput.files[0]); });
  uploadEl.addEventListener('dragover', function(e) { e.preventDefault(); uploadEl.classList.add('drag-over'); });
  uploadEl.addEventListener('dragleave', function() { uploadEl.classList.remove('drag-over'); });
  uploadEl.addEventListener('drop', function(e) { e.preventDefault(); uploadEl.classList.remove('drag-over'); if (e.dataTransfer.files[0]) processFile(e.dataTransfer.files[0]); });

  document.getElementById('enrich-download').addEventListener('click', function() {
    var rows = [['Original Name','Match','Matched Name','Firm','Stages','Sectors','Check Size','Location','TLDR']];
    enrichedRows.forEach(function(r) { rows.push(r.map(function(c){ return '"' + String(c).replace(/"/g,'""') + '"'; })); });
    var blob = new Blob([rows.map(function(r){return r.join(',');}).join('\\n')], {type:'text/csv'});
    var a = document.createElement('a'); a.href = URL.createObjectURL(blob); a.download = 'seedlist-enriched.csv'; a.click();
  });

  document.getElementById('enrich-reset').addEventListener('click', function() {
    uploadSection.style.display = 'block';
    previewSection.style.display = 'none';
    fileInput.value = '';
  });
})();
""")

@rt('/enrich')
def get():
    return Shell(
        Tool_Page(
            Page_Hero(H1('Enrich your investor list'),
                     P(f'Upload a CSV of investor names. We\'ll match them against {len(investors)} profiles and add Seedlist intelligence.'),
                     Sub_Link(A('Part of Founder Tools', href='/discover'))),
            Div(Upload_Area(
                    Span('📄', cls='icon'),
                    P(Span('Choose a CSV file', cls='cta'), ' or drag and drop'),
                    P('Accepts any CSV with investor or firm names'),
                    Input(type='file', id='file-input', accept='.csv,.tsv,.txt'),
                    id='upload-area'),
                Privacy_Note('Your data never leaves your browser.'),
                id='upload-section'),
            Div(Enrich_Stats(id='enrich-stats'),
                Enrich_Actions(
                    Button('Download enriched CSV', id='enrich-download', cls='btn-primary'),
                    Button('Upload another', id='enrich-reset', cls='btn-secondary')),
                Result_Table(Table(
                    Thead(Tr(Th('Original'), Th('Match'), Th('Matched Name'), Th('Firm'), Th('Stages'), Th('Check'), Th('TLDR'))),
                    Tbody(id='enrich-tbody'))),
                Alt_Tools('Try other tools: ', A('Build a target list', href='/find'), ' | ', A('Find by comparables', href='/comparables')),
                id='enrich-preview', style='display:none'),
            enrich_js),
        title='Enrich Your List')

## Routes — Match

In [ ]:
match_js = Script("""
(function() {
  var selectedSectors = new Set();
  var btns = document.querySelectorAll('#m-sector-grid .sector-btn');
  var hint = document.getElementById('m-sector-hint');
  var hidden = document.getElementById('m-sectors');
  btns.forEach(function(btn) {
    btn.addEventListener('click', function() {
      var v = btn.dataset.value;
      if (selectedSectors.has(v)) { selectedSectors.delete(v); btn.removeAttribute('active'); }
      else if (selectedSectors.size < 3) { selectedSectors.add(v); btn.setAttribute('active',''); }
      hint.textContent = selectedSectors.size + '/3 selected.';
      hidden.value = Array.from(selectedSectors).join(',');
    });
  });
})();
""")


In [ ]:
from starlette.datastructures import UploadFile
import csv, io, asyncio

def _csv_to_md(raw):
    "Parse CSV/TSV text into a markdown table string."
    sep = '\t' if '\t' in raw.split('\n')[0] else ','
    reader = csv.reader(io.StringIO(raw), delimiter=sep)
    rows = [r for r in reader if any(c.strip() for c in r)]
    if len(rows) < 2: return ''
    hdr = '| ' + ' | '.join(rows[0]) + ' |'
    div = '| ' + ' | '.join('---' for _ in rows[0]) + ' |'
    body = '\n'.join('| ' + ' | '.join(r) + ' |' for r in rows[1:])
    return f"{hdr}\n{div}\n{body}"

MATCH_DIR = Path('/app/data/matches')
MATCH_DIR.mkdir(parents=True, exist_ok=True)

@rt('/match')
async def post(session, company: str='', email: str='', stage: str='', raise_amount: str='', description: str='', website: str='', traction: str='', moat: str='', beyond_capital: str='', sectors: str='', existing_investors: str='', coinvest: str='', consent: str='', deck: UploadFile=None, investor_csv: UploadFile=None):
    if not session.get('auth'): return RedirectResponse(solve_signin_rt, status_code=303)
    err = lambda msg: Div(P(f'⚠️ {msg}'), id='m-warning', style='background:#fef2f2;color:#991b1b;padding:0.75rem 1rem;border-radius:var(--radius);margin-bottom:1rem;font-size:0.9rem')
    missing = []
    if not company: missing.append('Company name')
    if not email: missing.append('Email')
    if not stage: missing.append('Stage')
    if not raise_amount: missing.append('Raise amount')
    if not description: missing.append('Description')
    if not sectors: missing.append('Sectors')
    if not consent: missing.append('Consent checkbox')
    if missing: return err('Please fill in: ' + ', '.join(missing))

    slug = re.sub(r'[^a-z0-9]+', '-', company.lower()).strip('-')
    task_dir = MATCH_DIR/slug
    task_dir.mkdir(parents=True, exist_ok=True)

    csv_md = ''
    if investor_csv and investor_csv.filename:
        if not investor_csv.filename.lower().endswith(('.csv', '.tsv', '.txt')): return err('Investor file must be CSV, TSV, or TXT.')
        raw = (await investor_csv.read()).decode('utf-8', errors='replace')
        if len(raw) > 1_000_000: return err('Investor CSV must be under 1MB.')
        csv_md = _csv_to_md(raw)
        if not csv_md: return err('Could not parse investor CSV — needs a header row and at least one data row.')
        (task_dir/'investors.csv').write_text(raw)

    deck_path = None
    if deck and deck.filename:
        if not deck.filename.lower().endswith('.pdf'): return err('Pitch deck must be a PDF.')
        deck_bytes = await deck.read()
        if len(deck_bytes) > 10_000_000: return err('Pitch deck must be under 10MB.')
        deck_path = str(task_dir/'deck.pdf')
        Path(deck_path).write_bytes(deck_bytes)

    pitch = f"**Company:** {company}\n**Website:** {website or 'N/A'}\n**Stage:** {stage}\n**Raising:** {raise_amount}\n**Sectors:** {sectors}\n**Description:** {description}\n**Traction:** {traction or 'N/A'}\n**Defensibility:** {moat or 'N/A'}\n**Beyond capital:** {beyond_capital or 'N/A'}"
    if existing_investors.strip(): pitch += f"\n\n**Existing investors (avoid recommending):**\n{existing_investors.strip()}"
    if csv_md: pitch += f"\n\n**Founder's existing investor/outreach list (avoid recommending):**\n\n{csv_md}"
    if coinvest: pitch += f"\n\n**Open to Seedlist co-investing:** {coinvest}"

    (task_dir/'pitch.md').write_text(pitch)
    session['match_company'] = company
    session['match_email'] = email
    session['match_slug'] = slug

    task_id = f'{slug}-{uuid4().hex[:8]}'
    dname = f'/seedlist/matches/{task_id}'
    asyncio.create_task(run_match(dname, pitch, email, company, deck_path))

    return Response('', headers={'HX-Redirect': '/match/success'})


@rt('/match/success')
def get(session):
    company = session.get('match_company', 'your startup')
    email = session.get('match_email', 'your email')
    return Shell(
        Tool_Page(
            Page_Hero(H1('We\'re on it! 🚀'),
                     P(f'Our matching agent is analyzing {company} against {len(investors)} investor profiles.'),
                     P(f'Results will be sent to ', Strong(email), ' when ready — typically a few minutes.'))),
        title='Match Submitted')

In [ ]:
@rt('/match')
def get(session):
    if not session.get('auth'): return RedirectResponse(solve_signin_rt, status_code=303)
    sector_btns_m = [Span(parent_labels[k], cls='sector-btn', data_value=k) for k in taxonomy]
    return Shell(
        Tool_Page(
            Page_Hero(H1('Get matched with investors'),
                     P(f'Tell us about your startup and we\'ll find the best-fit investors from {len(investors)} profiles.')),
            Form(Tool_Form(
                Div(id='m-warning'),
                Label('Company name *', _for='m-company'),
                Input(type='text', name='company', id='m-company', placeholder='Acme Inc.'),
                Label('Your email *', _for='m-email'),
                Input(type='text', name='email', id='m-email', placeholder='founder@acme.com'),
                Label('Website', _for='m-website'),
                Input(type='text', name='website', id='m-website', placeholder='https://acme.com'),
                Label('Stage *', _for='m-stage'),
                Select(Option('Select stage...', value=''), Option('Pre-seed', value='pre-seed'),
                       Option('Seed', value='seed'), Option('Series A', value='series-a'),
                       Option('Series B+', value='series-b'), name='stage', id='m-stage'),
                Label('How much are you raising? *', _for='m-raise'),
                Input(type='text', name='raise_amount', id='m-raise', placeholder='$2M'),
                Label('Sectors *'),
                Div(*sector_btns_m, cls='sector-grid', id='m-sector-grid'),
                P('Pick 1 to 3 sectors. ', Span('0/3 selected.', id='m-sector-hint'), cls='hint'),
                Input(type='hidden', name='sectors', id='m-sectors'),
                Label('What does your company do? *', _for='m-desc'),
                Textarea(name='description', id='m-desc', placeholder='One or two sentences — what you do, for whom, and why it matters.', rows=3),
                Label('Strongest traction proof point', _for='m-traction'),
                Input(type='text', name='traction', id='m-traction', placeholder='$50K MRR, 10K users, LOIs from 3 enterprises…'),
                Label('What makes this hard / defensible?', _for='m-moat'),
                Textarea(name='moat', id='m-moat', placeholder='Why you, why now? 2-3 sentences.', rows=3),
                Label('What do you want beyond capital?', _for='m-beyond'),
                Input(type='text', name='beyond_capital', id='m-beyond', placeholder='Domain expertise, intros to customers, technical help…'),
                Label('Existing investors (we\'ll avoid recommending them)', _for='m-existing'),
                Textarea(name='existing_investors', id='m-existing', placeholder='Names of investors you already have or are talking to', rows=2),
                Label('Pitch deck (PDF, optional)', _for='m-deck'),
                Input(type='file', name='deck', id='m-deck', accept='.pdf'),
                Label('Existing investor list (CSV, optional)', _for='m-csv'),
                Input(type='file', name='investor_csv', id='m-csv', accept='.csv,.tsv,.txt'),
                Label('Open to Seedlist co-investing?', _for='m-coinvest'),
                Select(Option('No preference', value=''), Option('Yes', value='yes'), Option('No', value='no'), name='coinvest', id='m-coinvest'),
                Label(Input(type='checkbox', name='consent', value='yes', style='margin-right:0.5rem'),
                      'I consent to Seedlist sharing my submission with matched investors via Eric Ries',
                      style='font-weight:normal;font-size:0.9rem;display:flex;align-items:center;margin:1rem 0'),
                Button('Find my investors →', type='submit')),
                hx_post='/match', hx_target='#m-warning', hx_swap='outerHTML', enctype='multipart/form-data'),
            match_js),
        title='Get Matched')

## Server

In [ ]:
srv = JupyUvi(app)

## Routes — Homepage, Search & Discover

In [ ]:

@rt('/api/search-index.json')
def get():
    entries = []
    for p in investors:
        text = ' '.join(str(x) for x in [p.get('name',''), p.get('firm',''), p.get('role',''), p.get('location','')]
                        + (p.get('stage_focus') or []) + (p.get('sector_focus') or []) if x)
        entries.append(dict(name=p.get('name',''), type='investor', url=f"/investors/{p.get('slug','')}", firm=p.get('firm',''), location=p.get('location',''), text=text))
    for p in firms:
        text = ' '.join(str(x) for x in [p.get('name',''), p.get('location','')] if x)
        entries.append(dict(name=p.get('name',''), type='firm', url=f"/firms/{p.get('slug','')}", firm='', location=p.get('location',''), text=text))
    for p in startups:
        text = ' '.join(str(x) for x in [p.get('name',''), p.get('location','')] + (p.get('sector') or []) if x)
        entries.append(dict(name=p.get('name',''), type='startup', url=f"/startups/{p.get('slug','')}", firm='', location=p.get('location',''), text=text))
    return Response(_json.dumps(entries), media_type='application/json')

search_js = Script("""
(function() {
  var input = document.querySelector('search-box input');
  if (!input) return;
  var resultsEl = document.querySelector('search-results');
  var index = null;

  fetch('/api/search-index.json').then(function(r){return r.json();}).then(function(d){index=d;});

  input.addEventListener('input', function() {
    var q = input.value.trim().toLowerCase();
    if (!q || !index) { resultsEl.hidden = true; resultsEl.innerHTML = ''; return; }
    var terms = q.split(/\\s+/);
    var matches = index.filter(function(e) {
      var hay = e.text.toLowerCase();
      return terms.every(function(t) { return hay.indexOf(t) !== -1; });
    }).slice(0, 15);
    if (!matches.length) { resultsEl.innerHTML = '<div style="padding:0.75rem;color:var(--muted)">No results</div>'; resultsEl.hidden = false; return; }
    resultsEl.innerHTML = matches.map(function(m) {
      var badge = m.type === 'investor' ? '<span style="font-size:0.7rem;padding:0.1rem 0.4rem;border-radius:999px;background:#f5f3ff;color:#7c3aed;margin-left:0.5rem">Investor</span>'
        : m.type === 'firm' ? '<span style="font-size:0.7rem;padding:0.1rem 0.4rem;border-radius:999px;background:#ecfeff;color:#0891b2;margin-left:0.5rem">Firm</span>'
        : '<span style="font-size:0.7rem;padding:0.1rem 0.4rem;border-radius:999px;background:#fef3c7;color:#92400e;margin-left:0.5rem">Startup</span>';
      var sub = [m.firm, m.location].filter(Boolean).join(' · ');
      return '<a href="' + m.url + '" style="display:block;padding:0.5rem 0.75rem;color:var(--text);border-bottom:1px solid var(--border)">'
        + '<div style="font-weight:600;font-size:0.9rem">' + m.name + badge + '</div>'
        + (sub ? '<div style="font-size:0.8rem;color:var(--muted)">' + sub + '</div>' : '')
        + '</a>';
    }).join('');
    resultsEl.hidden = false;
  });

  document.addEventListener('click', function(e) { if (!resultsEl.contains(e.target) && e.target !== input) resultsEl.hidden = true; });
  input.addEventListener('focus', function() { if (resultsEl.innerHTML && input.value.trim()) resultsEl.hidden = false; });
})();
""")

@rt
def index():
    return Shell(
        Page_Hero(H1('Find the right investor for your startup'),
                 P(f'Browse {len(investors)} investors, {len(firms)} firms, and {len(startups)} startups. LLM-researched and regularly updated.'),
                 Search_Box(Input(type='text', placeholder='Search investors, firms, sectors…', autocomplete='off'),
                            Search_Results(hidden=True))),
        Card_Grid(A(Link_Card(H2('Investors'), P(f'{len(investors)} profiles with inferred theses')), href='/investors'),
                 A(Link_Card(H2('Firms'), P(f'{len(firms)} VC firm profiles')), href='/firms'),
                 A(Link_Card(H2('Groups'), P('Investors clustered by era, thesis, and network')), href='/groups')),
        Card_Grid(A(Link_Card(H2('Fundraising Tools'), P('Build a target list, enrich your CSV, or find by comparables')), href='/discover'),
                 A(Link_Card(H2('Round Feed'), P('Latest startup funding rounds')), href='/rounds'),
                 A(Link_Card(H2('Startups'), P(f'{len(startups)} portfolio companies tracked')), href='/startups')),
        Card_Grid(A(Link_Card(H2('For AI Agents'), P('JSON API, llms.txt, and a Claude Skill')), href='/agents')),
        search_js)

@rt('/discover')
def get():
    return Shell(
        Page_Hero(H1('Fundraising tools for founders'),
                 P('Three ways to build the right investor list. All free, all private — your data never leaves your browser.')),
        Card_Grid(A(Link_Card(H2('Enrich my list'), P('Upload a CSV of investor names and we\'ll add Seedlist intelligence.')), href='/enrich'),
                 A(Link_Card(H2('Build a target list'), P('Tell us your stage, sector, and check size and we\'ll find the best-fit investors.')), href='/find'),
                 A(Link_Card(H2('Find by comparables'), P('Pick startups similar to yours and we\'ll show you who invested in them.')), href='/comparables')),
        title='Fundraising Tools')

any secrets in here i should be worried about exposing?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I took a look through the code. No API keys or tokens are hardcoded in the served code, which is good. A few things worth noting though:

**Not exposed but worth being aware of:**
- `/app/data/credentials.json` and `/app/data/token.json` (Gmail OAuth) are referenced by path only — server-side, never served. Safe as long as the static file path (`/app/data/seedlist/static`) doesn't overlap.
- `erikgaas@gmail.com` is in the CRAFT's `run_match` pipeline (P5) — server-side only, never sent to browsers.

**Potentially worth tightening:**
- **Match submissions on disk** — `MATCH_DIR = Path('/app/data/seedlist/matches')` stores founder pitches (including email, company details, decks) as plaintext files. No access control beyond the filesystem. If the static file path ever overlapped or a directory listing route was added, these could leak.
- **No rate limiting on `/match` POST** — a logged-in user could spam submissions, each of which spawns an agent dialog + sends emails.
- **Session `auth` is just the `sub` claim** — no expiry check visible, so a session token could be long-lived.

**Intentionally public (just confirming you're OK with it):**
- All `/api/*.json` endpoints are unauthenticated — full investor data, startup-investor maps, search index. The agents page says "No auth, no rate limits" so this seems deliberate.

Nothing looks like an acute leak. The main thing I'd keep an eye on is that `matches/` directory staying isolated from anything publicly served.

<details class='token-usage-details'><summary>$0.3145</summary>

`total=70,028 | in=69,078 | out=950 | cached=51.3% | cache_new=5 | $0.3145`

</details>


yeah so MATCH_DIR should probably be /app/data/matches sound reasonable to you?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yeah, that's a sensible move — gets it out of the `seedlist/data` subtree entirely, away from anything the static file server could touch.

```python
MATCH_DIR = Path('/app/data/matches')
```

Also heads up — there's a tool error: `list_pyskills` not found. You may need to run it if you're using pyskills.

<details class='token-usage-details'><summary>$0.6549</summary>

`total=69,634 | in=69,542 | out=92 | cached=0.0% | cache_new=69,290 | $0.6549`

</details>
